dataset

In [ ]:
!pip -q install datasets pillow matplotlib


In [ ]:
!pip -q install pandas


In [ ]:
from google.colab import userdata
userdata.get('secretName')


In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token)


In [ ]:
from datasets import load_dataset

ds = load_dataset("sebnae/IndoorCrowd", "obj_det_seg")
ds


In [ ]:
print(ds)
print(ds.keys())

split_name = list(ds.keys())[0]
data = ds[split_name]

print(data.features)

sample = data[0]
print(sample.keys())
for k, v in sample.items():
    print(k, type(v))


In [ ]:
sample = ds["acs_ec"][0]

print("scene:", sample["scene"])
print("recording_id:", sample["recording_id"])
print("file_name:", sample["file_name"])
print("image_id:", sample["image_id"])
print("width x height:", sample["width"], sample["height"])

print("\nobjects keys:", sample["objects"].keys())
print("num boxes:", len(sample["objects"]["bbox"]))
print("first 5 boxes:", sample["objects"]["bbox"][:5])
print("first 5 track_ids:", sample["objects"]["track_id"][:5])
print("first 5 ids:", sample["objects"]["id"][:5])
print("first 5 scores:", sample["objects"]["score"][:5])


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

sample = ds["acs_ec"][0]
img = sample["image"]
objects = sample["objects"]

fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(img)

for box, tid in zip(objects["bbox"], objects["track_id"]):
    x, y, w, h = box
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.text(x, y - 5, f"tid={tid}", color="yellow", fontsize=9, backgroundcolor="black")

ax.axis("off")
plt.show()


In [ ]:
from collections import Counter

for split_name in ds.keys():
    data = ds[split_name]
    recs = [data[i]["recording_id"] for i in range(len(data))]
    counts = Counter(recs)
    print(f"\n=== {split_name} ===")
    print("num recordings:", len(counts))
    print("top recordings:", counts.most_common(10))


In [ ]:
data = ds["acs_ec"]

rows = []
for i in range(20):
    rows.append((data[i]["recording_id"], data[i]["image_id"], data[i]["file_name"]))

for row in rows:
    print(row)


In [ ]:
data = ds["acs_ec"]

# возьмем первые 50 кадров одного recording_id
target_recording = data[0]["recording_id"]
subset = [data[i] for i in range(len(data)) if data[i]["recording_id"] == target_recording]

# отсортируем по image_id
subset = sorted(subset, key=lambda x: x["image_id"])

for k in range(3):
    frame = subset[k]
    tids = frame["objects"]["track_id"]
    print(f"frame image_id={frame['image_id']}, num_people={len(tids)}, track_ids={tids[:10]}")


In [ ]:
for k in range(1, 5):
    prev_ids = set(subset[k-1]["objects"]["track_id"])
    curr_ids = set(subset[k]["objects"]["track_id"])
    inter = prev_ids.intersection(curr_ids)
    print(
        f"{subset[k-1]['image_id']} -> {subset[k]['image_id']}: "
        f"shared track_ids = {sorted(list(inter))[:10]}, count={len(inter)}"
    )


In [ ]:
from collections import defaultdict

for split_name in ds.keys():
    data = ds[split_name]
    frame_counts = defaultdict(int)
    people_counts = defaultdict(list)

    for i in range(len(data)):
        rec = data[i]["recording_id"]
        frame_counts[rec] += 1
        people_counts[rec].append(len(data[i]["objects"]["track_id"]))

    rows = []
    for rec in frame_counts:
        rows.append((
            rec,
            frame_counts[rec],
            sum(people_counts[rec]) / len(people_counts[rec])
        ))

    rows = sorted(rows, key=lambda x: x[1], reverse=True)

    print(f"\n=== {split_name} ===")
    for row in rows[:10]:
        print("recording:", row[0], "| frames:", row[1], "| avg people/frame:", round(row[2], 2))


In [ ]:
split_name = "acs_eg"
target_recording = "acs_eg_recording_1771861251_v2"

data = ds[split_name]
subset = [data[i] for i in range(len(data)) if data[i]["recording_id"] == target_recording]
subset = sorted(subset, key=lambda x: x["image_id"])

print("recording:", target_recording)
print("num frames:", len(subset))
print("first image_id:", subset[0]["image_id"])
print("last image_id:", subset[-1]["image_id"])
print("avg people/frame:", sum(len(f["objects"]["track_id"]) for f in subset) / len(subset))


In [ ]:
from collections import Counter

track_freq = Counter()
for frame in subset:
    track_freq.update(frame["objects"]["track_id"])

print("Top GT track_ids by frame count:")
for tid, cnt in track_freq.most_common(20):
    print("track_id:", tid, "| frames present:", cnt)


In [ ]:
top_tids = [tid for tid, cnt in track_freq.most_common(5)]
print("candidate GT track_ids:", top_tids)

for tid in top_tids:
    frames_with_tid = []
    for frame in subset:
        if tid in frame["objects"]["track_id"]:
            frames_with_tid.append(frame["image_id"])
    print(f"track_id={tid}: first={frames_with_tid[0]}, last={frames_with_tid[-1]}, count={len(frames_with_tid)}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

gt_tid = top_tids[0]  # потом можешь поменять

candidate_frames = []
for frame in subset:
    if gt_tid in frame["objects"]["track_id"]:
        candidate_frames.append(frame)

# возьмем 4 кадра по sequence
idxs = np.linspace(0, len(candidate_frames)-1, 4, dtype=int)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, idx in zip(axes, idxs):
    frame = candidate_frames[idx]
    img = frame["image"]
    tids = frame["objects"]["track_id"]
    bboxes = frame["objects"]["bbox"]

    ax.imshow(img)
    for box, tid in zip(bboxes, tids):
        if tid == gt_tid:
            x, y, w, h = box
            rect = patches.Rectangle((x, y), w, h, linewidth=3, edgecolor="red", facecolor="none")
            ax.add_patch(rect)
            ax.text(x, y - 5, f"GT tid={tid}", color="yellow", fontsize=10, backgroundcolor="black")
            break

    ax.set_title(f"image_id={frame['image_id']}")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
from collections import defaultdict
import numpy as np

tid_stats = defaultdict(lambda: {
    "count": 0,
    "areas": [],
    "widths": [],
    "heights": [],
    "center_x": [],
    "center_y": []
})

for frame in subset:
    for box, tid in zip(frame["objects"]["bbox"], frame["objects"]["track_id"]):
        x, y, w, h = box
        tid_stats[tid]["count"] += 1
        tid_stats[tid]["areas"].append(w * h)
        tid_stats[tid]["widths"].append(w)
        tid_stats[tid]["heights"].append(h)
        tid_stats[tid]["center_x"].append(x + w / 2)
        tid_stats[tid]["center_y"].append(y + h / 2)

rows = []
for tid, s in tid_stats.items():
    rows.append({
        "track_id": tid,
        "frames": s["count"],
        "mean_area": np.mean(s["areas"]),
        "mean_w": np.mean(s["widths"]),
        "mean_h": np.mean(s["heights"]),
        "mean_cx": np.mean(s["center_x"]),
        "mean_cy": np.mean(s["center_y"]),
    })

rows = sorted(rows, key=lambda r: (-r["frames"], -r["mean_area"]))

for r in rows[:30]:
    print(
        f"tid={r['track_id']:>3} | frames={r['frames']:>4} | "
        f"mean_area={r['mean_area']:.1f} | mean_w={r['mean_w']:.1f} | "
        f"mean_h={r['mean_h']:.1f} | mean_cx={r['mean_cx']:.1f} | mean_cy={r['mean_cy']:.1f}"
    )


In [ ]:
good = [
    r for r in rows
    if r["frames"] >= 100
    and r["mean_area"] >= 3000
    and 150 <= r["mean_cx"] <= 1130
]

print("good candidates:", len(good))
for r in good[:20]:
    print(
        f"tid={r['track_id']:>3} | frames={r['frames']:>4} | "
        f"mean_area={r['mean_area']:.1f} | mean_w={r['mean_w']:.1f} | "
        f"mean_h={r['mean_h']:.1f} | mean_cx={r['mean_cx']:.1f}"
    )


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def show_gt_track(subset, gt_tid, ncols=4):
    candidate_frames = []
    for frame in subset:
        if gt_tid in frame["objects"]["track_id"]:
            candidate_frames.append(frame)

    idxs = np.linspace(0, len(candidate_frames)-1, ncols, dtype=int)

    fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 5))
    if ncols == 1:
        axes = [axes]

    for ax, idx in zip(axes, idxs):
        frame = candidate_frames[idx]
        img = frame["image"]
        tids = frame["objects"]["track_id"]
        bboxes = frame["objects"]["bbox"]

        ax.imshow(img)
        for box, tid in zip(bboxes, tids):
            if tid == gt_tid:
                x, y, w, h = box
                rect = patches.Rectangle((x, y), w, h, linewidth=3, edgecolor="red", facecolor="none")
                ax.add_patch(rect)
                ax.text(x, y - 5, f"GT tid={tid}", color="yellow", fontsize=10, backgroundcolor="black")
                break

        ax.set_title(f"image_id={frame['image_id']}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
for tid in [r["track_id"] for r in good[:5]]:
    print("Showing GT track:", tid)
    show_gt_track(subset, tid)


In [ ]:
for tid in [2, 9, 10, 13]:
    print("Showing GT track:", tid)
    show_gt_track(subset, tid)


New tests of models and preparation

In [ ]:
from collections import defaultdict

PRIMARY_TID = 2
SECONDARY_TID = 10
OPTIONAL_HARD_TID = 13

TARGET_TIDS = [PRIMARY_TID, SECONDARY_TID]

gt_tracks = {tid: {} for tid in TARGET_TIDS}

for frame in subset:
    image_id = frame["image_id"]
    tids = frame["objects"]["track_id"]
    bboxes = frame["objects"]["bbox"]

    for box, tid in zip(bboxes, tids):
        if tid in gt_tracks:
            gt_tracks[tid][image_id] = box

for tid in TARGET_TIDS:
    print(f"GT tid={tid}: {len(gt_tracks[tid])} frames")


In [ ]:
def bbox_xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]

def iou_xywh(boxA, boxB):
    ax1, ay1, ax2, ay2 = bbox_xywh_to_xyxy(boxA)
    bx1, by1, bx2, by2 = bbox_xywh_to_xyxy(boxB)

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    areaA = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    areaB = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = areaA + areaB - inter_area
    if union <= 0:
        return 0.0

    return inter_area / union


In [ ]:
def match_gt_to_predictions(gt_box, pred_list, iou_threshold=0.3):
    """gt_box: [x, y, w, h]."""
    best_pred = None
    best_iou = 0.0

    for pred in pred_list:
        pred_box = pred["bbox"]
        iou = iou_xywh(gt_box, pred_box)
        if iou > best_iou:
            best_iou = iou
            best_pred = pred

    if best_iou >= iou_threshold:
        return True, best_pred, best_iou
    return False, None, best_iou


In [ ]:
from collections import Counter

def compute_longest_true_run(bool_list):
    best = 0
    cur = 0
    for v in bool_list:
        if v:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best

def compute_lost_segments(found_list):
    """
    lost segment = transition from found -> not found,
    counted as a lost episode that may later recover
    """lost_segments = 0."""
    pred_id_seq: list of predicted IDs for matched frames only, with None for unmatched
    Count switches only across consecutive matched detections.
    """switches = 0."""
    gt_track_dict: {image_id: gt_bbox}
    pred_by_image_id: {image_id: [ {"bbox":[...], "track_id":..., "score":...}, ... ]}

    returns dict with metrics + detailed per-frame records
    """
    image_ids = sorted(gt_track_dict.keys())

    found_list = []
    matched_ious = []
    pred_id_seq = []
    per_frame = []

    for image_id in image_ids:
        gt_box = gt_track_dict[image_id]
        pred_list = pred_by_image_id.get(image_id, [])

        matched, best_pred, best_iou = match_gt_to_predictions(
            gt_box, pred_list, iou_threshold=iou_threshold
        )

        if matched:
            found_list.append(True)
            matched_ious.append(best_iou)
            pred_id_seq.append(best_pred["track_id"])
            per_frame.append({
                "image_id": image_id,
                "gt_bbox": gt_box,
                "found": True,
                "iou": best_iou,
                "pred_track_id": best_pred["track_id"],
                "pred_bbox": best_pred["bbox"],
                "pred_score": best_pred.get("score", None),
            })
        else:
            found_list.append(False)
            pred_id_seq.append(None)
            per_frame.append({
                "image_id": image_id,
                "gt_bbox": gt_box,
                "found": False,
                "iou": best_iou,
                "pred_track_id": None,
                "pred_bbox": None,
                "pred_score": None,
            })

    total_frames = len(image_ids)
    matched_frames = sum(found_list)
    lock_rate = matched_frames / total_frames if total_frames > 0 else 0.0
    mean_iou = sum(matched_ious) / len(matched_ious) if matched_ious else 0.0

    longest_run = compute_longest_true_run(found_list)
    lost_segments = compute_lost_segments(found_list)
    id_switches = compute_id_switches(pred_id_seq)

    assigned_ids = [pid for pid in pred_id_seq if pid is not None]
    unique_assigned_ids = len(set(assigned_ids))

    if assigned_ids:
        id_counter = Counter(assigned_ids)
        dominant_id, dominant_count = id_counter.most_common(1)[0]
        dominant_id_purity = dominant_count / len(assigned_ids)
    else:
        dominant_id = None
        dominant_id_purity = 0.0

    return {
        "total_gt_frames": total_frames,
        "matched_frames": matched_frames,
        "lock_rate": lock_rate,
        "mean_iou": mean_iou,
        "lost_segments": lost_segments,
        "longest_continuous_run": longest_run,
        "id_switches": id_switches,
        "unique_assigned_pred_ids": unique_assigned_ids,
        "dominant_pred_id": dominant_id,
        "dominant_id_purity": dominant_id_purity,
        "per_frame": per_frame,
    }


In [ ]:
PRIMARY_TID = 2
SECONDARY_TID = 10

TARGET_TIDS = [PRIMARY_TID, SECONDARY_TID]

gt_tracks = {tid: {} for tid in TARGET_TIDS}

for frame in subset:
    image_id = frame["image_id"]
    tids = frame["objects"]["track_id"]
    bboxes = frame["objects"]["bbox"]

    for box, tid in zip(bboxes, tids):
        if tid in gt_tracks:
            gt_tracks[tid][image_id] = box

for tid in TARGET_TIDS:
    print(f"GT tid={tid}: {len(gt_tracks[tid])} frames")


In [ ]:
import os
import cv2
import numpy as np

split_name = "acs_eg"
target_recording = "acs_eg_recording_1771861251_v2"

data = ds[split_name]
subset = [data[i] for i in range(len(data)) if data[i]["recording_id"] == target_recording]
subset = sorted(subset, key=lambda x: x["image_id"])

print("frames in subset:", len(subset))

frames_dir = "/content/indoorcrowd_seq_frames"
os.makedirs(frames_dir, exist_ok=True)

for frame in subset:
    img = np.array(frame["image"])  # RGB
    out_path = os.path.join(frames_dir, f"{frame['image_id']:05d}.jpg")
    cv2.imwrite(out_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

# Собираем mp4
frame_files = sorted([os.path.join(frames_dir, f) for f in os.listdir(frames_dir) if f.endswith(".jpg")])

img0 = cv2.imread(frame_files[0])
h, w = img0.shape[:2]

indoor_video_path = "/content/indoorcrowd_raw.mp4"
writer = cv2.VideoWriter(indoor_video_path, cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))

for f in frame_files:
    frame = cv2.imread(f)
    writer.write(frame)

writer.release()

print("Saved video:", indoor_video_path)
print("Resolution:", w, "x", h)


In [ ]:
PRIMARY_TID = 2
SECONDARY_TID = 10
TARGET_TIDS = [PRIMARY_TID, SECONDARY_TID]

gt_tracks = {tid: {} for tid in TARGET_TIDS}

for frame in subset:
    image_id = frame["image_id"]
    tids = frame["objects"]["track_id"]
    bboxes = frame["objects"]["bbox"]

    for box, tid in zip(bboxes, tids):
        if tid in gt_tracks:
            gt_tracks[tid][image_id] = box

for tid in TARGET_TIDS:
    print(f"GT tid={tid}: {len(gt_tracks[tid])} frames")


In [ ]:

INDOOR_RAW_VIDEO = "/content/indoorcrowd_raw.mp4"

INDOOR_STAB_DIR = "/content/indoorcrowd_v2"
os.makedirs(INDOOR_STAB_DIR, exist_ok=True)

results_indoor_v2 = stabilize_v2(
    input_path=INDOOR_RAW_VIDEO,
    output_path=f"{INDOOR_STAB_DIR}/indoorcrowd_v2_stabilized.mp4",
    side_by_side_path=f"{INDOOR_STAB_DIR}/indoorcrowd_v2_sbs.mp4",
    transforms_path=f"{INDOOR_STAB_DIR}/indoorcrowd_v2_transforms.npy",
    raw_transforms_path=f"{INDOOR_STAB_DIR}/indoorcrowd_v2_raw_transforms.npy",
    trajectory_path=f"{INDOOR_STAB_DIR}/indoorcrowd_v2_trajectory.npy",
    smoothed_trajectory_path=f"{INDOOR_STAB_DIR}/indoorcrowd_v2_smoothed_trajectory.npy",
    q=(1e-3, 1e-3, 1e-5),
    r=(10.0, 10.0, 1e-2),
    kalman_mode='rts',
    min_points=20,
    median_kernel=5,
    mad_k=4.0,
    verbose=True
)

INDOOR_STAB_VIDEO = f"{INDOOR_STAB_DIR}/indoorcrowd_v2_stabilized.mp4"
print("stabilized video:", INDOOR_STAB_VIDEO)


In [ ]:
import os

OUT_DIR_INDOOR = "/content/results_indoorcrowd_perception"
os.makedirs(OUT_DIR_INDOOR, exist_ok=True)

print(">>> RAW IndoorCrowd")
sum_raw_indoor, tracks_raw_indoor = detect_and_track(
    input_video=INDOOR_RAW_VIDEO,
    output_video=f"{OUT_DIR_INDOOR}/indoor_raw_tracked.mp4",
    imgsz=960,
    conf=0.20,
    verbose=True
)

print("\n>>> STABILIZED IndoorCrowd")
sum_stab_indoor, tracks_stab_indoor = detect_and_track(
    input_video=INDOOR_STAB_VIDEO,
    output_video=f"{OUT_DIR_INDOOR}/indoor_stab_tracked.mp4",
    imgsz=960,
    conf=0.20,
    verbose=True
)


In [ ]:
def tracks_log_to_pred_by_image_id(tracks_log):
    pred_by_image_id = {}

    for frame_entry in tracks_log:
        frame_idx = frame_entry["frame"]      # 0-based
        image_id = frame_idx + 1              # IndoorCrowd image_id is 1-based

        preds = []
        for tr in frame_entry["tracks"]:
            x1, y1, x2, y2 = tr["bbox"]
            preds.append({
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "track_id": int(tr["id"]),
                "score": float(tr["conf"]),
            })

        pred_by_image_id[image_id] = preds

    return pred_by_image_id

raw_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_raw_indoor)
stab_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_stab_indoor)

print("raw frames:", len(raw_preds_by_image_id))
print("stab frames:", len(stab_preds_by_image_id))

first_key = sorted(raw_preds_by_image_id.keys())[0]
print("example raw preds, image_id =", first_key)
print(raw_preds_by_image_id[first_key][:3])


In [ ]:
results = {}

for target_tid in TARGET_TIDS:
    gt_dict = gt_tracks[target_tid]

    results[(target_tid, "raw_ocsort")] = evaluate_gt_target(
        gt_dict,
        raw_preds_by_image_id,
        iou_threshold=0.3
    )

    results[(target_tid, "stab_ocsort")] = evaluate_gt_target(
        gt_dict,
        stab_preds_by_image_id,
        iou_threshold=0.3
    )

print("done")


In [ ]:
import pandas as pd

rows = []
for (target_tid, pipeline_name), res in results.items():
    rows.append({
        "target_tid": target_tid,
        "pipeline": pipeline_name,
        "gt_frames": res["total_gt_frames"],
        "matched_frames": res["matched_frames"],
        "lock_rate": round(res["lock_rate"], 4),
        "mean_iou": round(res["mean_iou"], 4),
        "lost_segments": res["lost_segments"],
        "longest_run": res["longest_continuous_run"],
        "id_switches": res["id_switches"],
        "unique_pred_ids": res["unique_assigned_pred_ids"],
        "dominant_pred_id": res["dominant_pred_id"],
        "dominant_id_purity": round(res["dominant_id_purity"], 4),
    })

df_results = pd.DataFrame(rows).sort_values(["target_tid", "pipeline"])
df_results


In [ ]:
import cv2
import numpy as np
import os
import math
import random

def add_realistic_shake(
    input_video,
    output_video,
    sin_amp_trans=10.0,
    sin_amp_rot=0.015,
    freqs=(2.5, 4.0, 7.0),
    white_std=1.5,
    burst_prob=0.015,
    burst_mag=12.0,
    seed=42,
    fps_override=None,
    border_mode=cv2.BORDER_REPLICATE,
):
    random.seed(seed)
    np.random.seed(seed)

    cap = cv2.VideoCapture(input_video)
    assert cap.isOpened(), f"Cannot open {input_video}"

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps_override is not None:
        fps = fps_override
    if fps <= 0:
        fps = 10.0

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    t = 0
    burst_dx = 0.0
    burst_dy = 0.0
    burst_da = 0.0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        time_s = t / fps

        # multi-frequency smooth vibration
        dx = 0.0
        dy = 0.0
        da = 0.0

        for i, f in enumerate(freqs):
            phase1 = 0.7 * i
            phase2 = 1.1 * i
            phase3 = 0.9 * i

            dx += (sin_amp_trans / (i + 1)) * math.sin(2 * math.pi * f * time_s + phase1)
            dy += (0.8 * sin_amp_trans / (i + 1)) * math.sin(2 * math.pi * (f * 0.9) * time_s + phase2)
            da += (sin_amp_rot / (i + 1)) * math.sin(2 * math.pi * (f * 1.1) * time_s + phase3)

        # small random noise
        dx += np.random.normal(0, white_std)
        dy += np.random.normal(0, white_std)
        da += np.random.normal(0, sin_amp_rot * 0.15)

        # occasional burst events with decay
        if random.random() < burst_prob:
            burst_dx += np.random.uniform(-burst_mag, burst_mag)
            burst_dy += np.random.uniform(-burst_mag, burst_mag)
            burst_da += np.random.uniform(-sin_amp_rot * 3, sin_amp_rot * 3)

        dx += burst_dx
        dy += burst_dy
        da += burst_da

        burst_dx *= 0.85
        burst_dy *= 0.85
        burst_da *= 0.85

        cx, cy = w / 2.0, h / 2.0
        cos_a = math.cos(da)
        sin_a = math.sin(da)

        M = np.array([
            [cos_a, -sin_a, dx + cx - cos_a * cx + sin_a * cy],
            [sin_a,  cos_a, dy + cy - sin_a * cx - cos_a * cy]
        ], dtype=np.float32)

        shaken = cv2.warpAffine(frame, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=border_mode)
        writer.write(shaken)
        t += 1

    cap.release()
    writer.release()
    print("Saved shaken video:", output_video)


In [ ]:
INDOOR_SHAKE_DIR = "/content/indoorcrowd_shake"
os.makedirs(INDOOR_SHAKE_DIR, exist_ok=True)

INDOOR_SHAKEN_VIDEO = f"{INDOOR_SHAKE_DIR}/indoorcrowd_shaken.mp4"

add_realistic_shake(
    input_video=INDOOR_RAW_VIDEO,
    output_video=INDOOR_SHAKEN_VIDEO,
    sin_amp_trans=10.0,
    sin_amp_rot=0.015,
    freqs=(2.5, 4.0, 7.0),
    white_std=1.5,
    burst_prob=0.015,
    burst_mag=12.0,
    seed=42,
    fps_override=10.0,
)

print("exists:", os.path.exists(INDOOR_SHAKEN_VIDEO))


In [ ]:
import inspect
print(inspect.signature(stabilize_v2))


In [ ]:
INDOOR_SHAKE_STAB_DIR = "/content/indoorcrowd_shake_v2"
os.makedirs(INDOOR_SHAKE_STAB_DIR, exist_ok=True)

INDOOR_SHAKEN_STAB_VIDEO = f"{INDOOR_SHAKE_STAB_DIR}/indoorcrowd_shaken_v2_stabilized.mp4"

results_indoor_shake_v2 = stabilize_v2(
    input_path=INDOOR_SHAKEN_VIDEO,
    output_path=INDOOR_SHAKEN_STAB_VIDEO,
    side_by_side_path=f"{INDOOR_SHAKE_STAB_DIR}/indoorcrowd_shaken_v2_sbs.mp4",
    transforms_path=f"{INDOOR_SHAKE_STAB_DIR}/indoorcrowd_shaken_v2_transforms.npy",
    raw_transforms_path=f"{INDOOR_SHAKE_STAB_DIR}/indoorcrowd_shaken_v2_raw_transforms.npy",
    trajectory_path=f"{INDOOR_SHAKE_STAB_DIR}/indoorcrowd_shaken_v2_trajectory.npy",
    smoothed_trajectory_path=f"{INDOOR_SHAKE_STAB_DIR}/indoorcrowd_shaken_v2_smoothed_trajectory.npy",
    q=(1e-3, 1e-3, 1e-5),
    r=(10.0, 10.0, 1e-2),
    kalman_mode='rts',
    min_points=20,
    median_kernel=5,
    mad_k=4.0,
    verbose=True
)

print("shaken stabilized exists:", os.path.exists(INDOOR_SHAKEN_STAB_VIDEO))
print("video:", INDOOR_SHAKEN_STAB_VIDEO)


In [ ]:
import os

OUT_DIR_SHAKEN = "/content/results_indoorcrowd_shaken_perception"
os.makedirs(OUT_DIR_SHAKEN, exist_ok=True)

print(">>> SHAKEN IndoorCrowd")
sum_shake_indoor, tracks_shake_indoor = detect_and_track(
    input_video=INDOOR_SHAKEN_VIDEO,
    output_video=f"{OUT_DIR_SHAKEN}/indoor_shaken_tracked.mp4",
    imgsz=960,
    conf=0.20,
    verbose=True
)

print("\n>>> SHAKEN + STABILIZED IndoorCrowd")
sum_shake_stab_indoor, tracks_shake_stab_indoor = detect_and_track(
    input_video=INDOOR_SHAKEN_STAB_VIDEO,
    output_video=f"{OUT_DIR_SHAKEN}/indoor_shaken_stab_tracked.mp4",
    imgsz=960,
    conf=0.20,
    verbose=True
)


In [ ]:
def tracks_log_to_pred_by_image_id(tracks_log):
    pred_by_image_id = {}

    for frame_entry in tracks_log:
        frame_idx = frame_entry["frame"]
        image_id = frame_idx + 1   # важно: GT image_id начинается с 1

        preds = []
        for tr in frame_entry["tracks"]:
            x1, y1, x2, y2 = tr["bbox"]
            preds.append({
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "track_id": int(tr["id"]),
                "score": float(tr["conf"]),
            })

        pred_by_image_id[image_id] = preds

    return pred_by_image_id

shake_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_shake_indoor)
shake_stab_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_shake_stab_indoor)

print("shake frames:", len(shake_preds_by_image_id))
print("shake_stab frames:", len(shake_stab_preds_by_image_id))
print("example shake preds:", shake_preds_by_image_id[1][:3])


In [ ]:
results_all = {}

for target_tid in TARGET_TIDS:
    gt_dict = gt_tracks[target_tid]

    results_all[(target_tid, "orig_ocsort")] = evaluate_gt_target(
        gt_dict, raw_preds_by_image_id, iou_threshold=0.3
    )

    results_all[(target_tid, "orig_stab_ocsort")] = evaluate_gt_target(
        gt_dict, stab_preds_by_image_id, iou_threshold=0.3
    )

    results_all[(target_tid, "shake_ocsort")] = evaluate_gt_target(
        gt_dict, shake_preds_by_image_id, iou_threshold=0.3
    )

    results_all[(target_tid, "shake_stab_ocsort")] = evaluate_gt_target(
        gt_dict, shake_stab_preds_by_image_id, iou_threshold=0.3
    )

print("done")


In [ ]:
import pandas as pd

rows = []
for (target_tid, pipeline_name), res in results_all.items():
    rows.append({
        "target_tid": target_tid,
        "pipeline": pipeline_name,
        "gt_frames": res["total_gt_frames"],
        "matched_frames": res["matched_frames"],
        "lock_rate": round(res["lock_rate"], 4),
        "mean_iou": round(res["mean_iou"], 4),
        "lost_segments": res["lost_segments"],
        "longest_run": res["longest_continuous_run"],
        "id_switches": res["id_switches"],
        "unique_pred_ids": res["unique_assigned_pred_ids"],
        "dominant_pred_id": res["dominant_pred_id"],
        "dominant_id_purity": round(res["dominant_id_purity"], 4),
    })

df_results_all = pd.DataFrame(rows).sort_values(["target_tid", "pipeline"])
df_results_all


новые методы трекинга

In [ ]:
from ultralytics import YOLO
import os

model = YOLO("yolo11n.pt")

print("Ultralytics model loaded.")


In [ ]:
import cv2
import json
import os
from ultralytics import YOLO

def detect_and_track_botsort(
    input_video,
    output_video,
    imgsz=960,
    conf=0.20,
    iou=0.45,
    tracker_yaml="botsort.yaml",
    verbose=True,
):
    model = YOLO("yolo11n.pt")

    cap = cv2.VideoCapture(input_video)
    assert cap.isOpened(), f"Cannot open video: {input_video}"

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 10.0

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    frame_idx = 0
    tracks_log = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(
            source=frame,
            persist=True,
            tracker=tracker_yaml,
            imgsz=imgsz,
            conf=conf,
            iou=iou,
            verbose=False,
            classes=[0],   # person only
        )

        r = results[0]
        frame_tracks = []

        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy() if r.boxes.conf is not None else None
            ids = r.boxes.id.cpu().numpy() if r.boxes.id is not None else None
            clss = r.boxes.cls.cpu().numpy() if r.boxes.cls is not None else None

            for i in range(len(xyxy)):
                if clss is not None and int(clss[i]) != 0:
                    continue
                if ids is None:
                    continue

                x1, y1, x2, y2 = xyxy[i]
                tid = int(ids[i])
                tconf = float(confs[i]) if confs is not None else 1.0

                frame_tracks.append({
                    "id": tid,
                    "bbox": [float(x1), float(y1), float(x2), float(y2)],
                    "conf": tconf,
                })

                # draw
                cv2.rectangle(
                    frame,
                    (int(x1), int(y1)),
                    (int(x2), int(y2)),
                    (0, 255, 0),
                    2
                )
                cv2.putText(
                    frame,
                    f"id={tid} {tconf:.2f}",
                    (int(x1), max(20, int(y1) - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 0),
                    1,
                    cv2.LINE_AA
                )

        writer.write(frame)

        tracks_log.append({
            "frame": frame_idx,
            "tracks": frame_tracks
        })

        frame_idx += 1
        if verbose and frame_idx % 100 == 0:
            print(f"[BoT-SORT] processed {frame_idx}/{n_frames}")

    cap.release()
    writer.release()

    out_json = output_video.replace(".mp4", "_tracks.json")
    with open(out_json, "w") as f:
        json.dump(tracks_log, f)

    summary = {
        "input_video": input_video,
        "output_video": output_video,
        "tracks_json": out_json,
        "fps": fps,
        "width": w,
        "height": h,
        "frames": frame_idx,
        "tracker": tracker_yaml,
    }

    if verbose:
        print("[BoT-SORT] done")
        print(summary)

    return summary, tracks_log


In [ ]:
import os

OUT_DIR_BOTSORT = "/content/results_indoorcrowd_botsort"
os.makedirs(OUT_DIR_BOTSORT, exist_ok=True)

print(">>> ORIG + BoT-SORT")
sum_orig_bot, tracks_orig_bot = detect_and_track_botsort(
    input_video=INDOOR_RAW_VIDEO,
    output_video=f"{OUT_DIR_BOTSORT}/indoor_orig_botsort.mp4",
    imgsz=960,
    conf=0.20,
    tracker_yaml="botsort.yaml",
    verbose=True
)

print("\n>>> ORIG_STAB + BoT-SORT")
sum_orig_stab_bot, tracks_orig_stab_bot = detect_and_track_botsort(
    input_video=INDOOR_STAB_VIDEO,
    output_video=f"{OUT_DIR_BOTSORT}/indoor_orig_stab_botsort.mp4",
    imgsz=960,
    conf=0.20,
    tracker_yaml="botsort.yaml",
    verbose=True
)

print("\n>>> SHAKE + BoT-SORT")
sum_shake_bot, tracks_shake_bot = detect_and_track_botsort(
    input_video=INDOOR_SHAKEN_VIDEO,
    output_video=f"{OUT_DIR_BOTSORT}/indoor_shake_botsort.mp4",
    imgsz=960,
    conf=0.20,
    tracker_yaml="botsort.yaml",
    verbose=True
)

print("\n>>> SHAKE_STAB + BoT-SORT")
sum_shake_stab_bot, tracks_shake_stab_bot = detect_and_track_botsort(
    input_video=INDOOR_SHAKEN_STAB_VIDEO,
    output_video=f"{OUT_DIR_BOTSORT}/indoor_shake_stab_botsort.mp4",
    imgsz=960,
    conf=0.20,
    tracker_yaml="botsort.yaml",
    verbose=True
)


In [ ]:
def tracks_log_to_pred_by_image_id(tracks_log):
    pred_by_image_id = {}

    for frame_entry in tracks_log:
        frame_idx = frame_entry["frame"]
        image_id = frame_idx + 1

        preds = []
        for tr in frame_entry["tracks"]:
            x1, y1, x2, y2 = tr["bbox"]
            preds.append({
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                "track_id": int(tr["id"]),
                "score": float(tr["conf"]),
            })

        pred_by_image_id[image_id] = preds

    return pred_by_image_id

orig_bot_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_orig_bot)
orig_stab_bot_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_orig_stab_bot)
shake_bot_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_shake_bot)
shake_stab_bot_preds_by_image_id = tracks_log_to_pred_by_image_id(tracks_shake_stab_bot)


In [ ]:
results_bot = {}

for target_tid in TARGET_TIDS:
    gt_dict = gt_tracks[target_tid]

    results_bot[(target_tid, "orig_botsort")] = evaluate_gt_target(
        gt_dict, orig_bot_preds_by_image_id, iou_threshold=0.3
    )

    results_bot[(target_tid, "orig_stab_botsort")] = evaluate_gt_target(
        gt_dict, orig_stab_bot_preds_by_image_id, iou_threshold=0.3
    )

    results_bot[(target_tid, "shake_botsort")] = evaluate_gt_target(
        gt_dict, shake_bot_preds_by_image_id, iou_threshold=0.3
    )

    results_bot[(target_tid, "shake_stab_botsort")] = evaluate_gt_target(
        gt_dict, shake_stab_bot_preds_by_image_id, iou_threshold=0.3
    )

print("done")


In [ ]:
import pandas as pd

rows_bot = []
for (target_tid, pipeline_name), res in results_bot.items():
    rows_bot.append({
        "target_tid": target_tid,
        "pipeline": pipeline_name,
        "gt_frames": res["total_gt_frames"],
        "matched_frames": res["matched_frames"],
        "lock_rate": round(res["lock_rate"], 4),
        "mean_iou": round(res["mean_iou"], 4),
        "lost_segments": res["lost_segments"],
        "longest_run": res["longest_continuous_run"],
        "id_switches": res["id_switches"],
        "unique_pred_ids": res["unique_assigned_pred_ids"],
        "dominant_pred_id": res["dominant_pred_id"],
        "dominant_id_purity": round(res["dominant_id_purity"], 4),
    })

df_results_bot = pd.DataFrame(rows_bot).sort_values(["target_tid", "pipeline"])
df_results_bot


In [ ]:
def get_res_bot(target_tid, name):
    return results_bot[(target_tid, name)]

def summarize_target_bot(target_tid):
    orig = get_res_bot(target_tid, "orig_botsort")
    orig_stab = get_res_bot(target_tid, "orig_stab_botsort")
    shake = get_res_bot(target_tid, "shake_botsort")
    shake_stab = get_res_bot(target_tid, "shake_stab_botsort")

    return {
        "target_tid": target_tid,

        "orig_lock_rate": round(orig["lock_rate"], 4),
        "orig_stab_lock_rate": round(orig_stab["lock_rate"], 4),
        "shake_lock_rate": round(shake["lock_rate"], 4),
        "shake_stab_lock_rate": round(shake_stab["lock_rate"], 4),

        "orig_lost_segments": orig["lost_segments"],
        "orig_stab_lost_segments": orig_stab["lost_segments"],
        "shake_lost_segments": shake["lost_segments"],
        "shake_stab_lost_segments": shake_stab["lost_segments"],

        "orig_longest_run": orig["longest_continuous_run"],
        "orig_stab_longest_run": orig_stab["longest_continuous_run"],
        "shake_longest_run": shake["longest_continuous_run"],
        "shake_stab_longest_run": shake_stab["longest_continuous_run"],

        "orig_id_switches": orig["id_switches"],
        "orig_stab_id_switches": orig_stab["id_switches"],
        "shake_id_switches": shake["id_switches"],
        "shake_stab_id_switches": shake_stab["id_switches"],

        "delta_orig_stab_lock_rate": round(orig_stab["lock_rate"] - orig["lock_rate"], 4),
        "delta_shake_recovery_lock_rate": round(shake_stab["lock_rate"] - shake["lock_rate"], 4),

        "delta_orig_stab_lost_segments": orig_stab["lost_segments"] - orig["lost_segments"],
        "delta_shake_recovery_lost_segments": shake_stab["lost_segments"] - shake["lost_segments"],

        "delta_orig_stab_id_switches": orig_stab["id_switches"] - orig["id_switches"],
        "delta_shake_recovery_id_switches": shake_stab["id_switches"] - shake["id_switches"],
    }

compare_4_bot_df = pd.DataFrame([summarize_target_bot(tid) for tid in TARGET_TIDS])
compare_4_bot_df


In [ ]:
df_results_all["tracker"] = "OC-SORT"
df_results_bot["tracker"] = "BoT-SORT"

df_combined = pd.concat([df_results_all, df_results_bot], ignore_index=True)
df_combined = df_combined.sort_values(["target_tid", "tracker", "pipeline"])
df_combined


In [ ]:
import pandas as pd
import os

EXPORT_DIR = "/content/thesis_final_exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

main_cols = [
    "target_tid", "pipeline", "lock_rate", "mean_iou",
    "lost_segments", "longest_run", "id_switches", "dominant_id_purity"
]

df_ocsort_main = (
    df_results_all[
        ["target_tid", "pipeline", "lock_rate", "mean_iou",
         "lost_segments", "longest_run", "id_switches", "dominant_id_purity"]
    ]
    .copy()
    .sort_values(["target_tid", "pipeline"])
)

display(df_ocsort_main)

df_ocsort_main.to_csv(f"{EXPORT_DIR}/table_ocsort_main_results.csv", index=False)
print("saved:", f"{EXPORT_DIR}/table_ocsort_main_results.csv")


In [ ]:
df_ocsort_pivot = df_ocsort_main.pivot(index="target_tid", columns="pipeline")
display(df_ocsort_pivot)

df_ocsort_pivot.to_csv(f"{EXPORT_DIR}/table_ocsort_main_results_pivot.csv")
print("saved:", f"{EXPORT_DIR}/table_ocsort_main_results_pivot.csv")


In [ ]:
df_tracker_sensitivity = df_combined[
    df_combined["pipeline"].isin(["shake_ocsort", "shake_stab_ocsort", "shake_botsort", "shake_stab_botsort"])
][[
    "target_tid", "tracker", "pipeline", "lock_rate", "mean_iou",
    "lost_segments", "longest_run", "id_switches", "dominant_id_purity"
]].copy()

display(df_tracker_sensitivity)

df_tracker_sensitivity.to_csv(f"{EXPORT_DIR}/table_tracker_sensitivity.csv", index=False)
print("saved:", f"{EXPORT_DIR}/table_tracker_sensitivity.csv")


In [ ]:
def get_res(target_tid, name):
    return results_all[(target_tid, name)]

summary_rows = []
for tid in TARGET_TIDS:
    orig = get_res(tid, "orig_ocsort")
    orig_stab = get_res(tid, "orig_stab_ocsort")
    shake = get_res(tid, "shake_ocsort")
    shake_stab = get_res(tid, "shake_stab_ocsort")

    summary_rows.append({
        "target_tid": tid,
        "orig_lock_rate": round(orig["lock_rate"], 4),
        "orig_stab_lock_rate": round(orig_stab["lock_rate"], 4),
        "shake_lock_rate": round(shake["lock_rate"], 4),
        "shake_stab_lock_rate": round(shake_stab["lock_rate"], 4),

        "orig_lost_segments": orig["lost_segments"],
        "orig_stab_lost_segments": orig_stab["lost_segments"],
        "shake_lost_segments": shake["lost_segments"],
        "shake_stab_lost_segments": shake_stab["lost_segments"],

        "orig_longest_run": orig["longest_continuous_run"],
        "orig_stab_longest_run": orig_stab["longest_continuous_run"],
        "shake_longest_run": shake["longest_continuous_run"],
        "shake_stab_longest_run": shake_stab["longest_continuous_run"],

        "orig_id_switches": orig["id_switches"],
        "orig_stab_id_switches": orig_stab["id_switches"],
        "shake_id_switches": shake["id_switches"],
        "shake_stab_id_switches": shake_stab["id_switches"],
    })

df_ocsort_summary = pd.DataFrame(summary_rows)
display(df_ocsort_summary)

df_ocsort_summary.to_csv(f"{EXPORT_DIR}/table_ocsort_summary_4modes.csv", index=False)
print("saved:", f"{EXPORT_DIR}/table_ocsort_summary_4modes.csv")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pipelines_order = ["orig_ocsort", "orig_stab_ocsort", "shake_ocsort", "shake_stab_ocsort"]
pipeline_labels = ["Original", "Original + V2", "Shaken", "Shaken + V2"]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(pipelines_order))
width = 0.35

vals_t2 = [results_all[(2, p)]["lost_segments"] for p in pipelines_order]
vals_t10 = [results_all[(10, p)]["lost_segments"] for p in pipelines_order]

ax.bar(x - width/2, vals_t2, width, label="Target 2")
ax.bar(x + width/2, vals_t10, width, label="Target 10")

ax.set_title("OC-SORT: Lost segments across four evaluation regimes")
ax.set_ylabel("Lost segments")
ax.set_xticks(x)
ax.set_xticklabels(pipeline_labels, rotation=15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{EXPORT_DIR}/fig_ocsort_lost_segments.png", dpi=300, bbox_inches="tight")
plt.show()

print("saved:", f"{EXPORT_DIR}/fig_ocsort_lost_segments.png")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

vals_t2 = [results_all[(2, p)]["lock_rate"] for p in pipelines_order]
vals_t10 = [results_all[(10, p)]["lock_rate"] for p in pipelines_order]

ax.bar(x - width/2, vals_t2, width, label="Target 2")
ax.bar(x + width/2, vals_t10, width, label="Target 10")

ax.set_title("OC-SORT: Lock rate across four evaluation regimes")
ax.set_ylabel("Lock rate")
ax.set_xticks(x)
ax.set_xticklabels(pipeline_labels, rotation=15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{EXPORT_DIR}/fig_ocsort_lock_rate.png", dpi=300, bbox_inches="tight")
plt.show()

print("saved:", f"{EXPORT_DIR}/fig_ocsort_lock_rate.png")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

vals_t2 = [results_all[(2, p)]["id_switches"] for p in pipelines_order]
vals_t10 = [results_all[(10, p)]["id_switches"] for p in pipelines_order]

ax.bar(x - width/2, vals_t2, width, label="Target 2")
ax.bar(x + width/2, vals_t10, width, label="Target 10")

ax.set_title("OC-SORT: ID switches across four evaluation regimes")
ax.set_ylabel("ID switches")
ax.set_xticks(x)
ax.set_xticklabels(pipeline_labels, rotation=15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{EXPORT_DIR}/fig_ocsort_id_switches.png", dpi=300, bbox_inches="tight")
plt.show()

print("saved:", f"{EXPORT_DIR}/fig_ocsort_id_switches.png")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

labels = ["Target 2\nShake", "Target 2\nShake+V2", "Target 10\nShake", "Target 10\nShake+V2"]

oc_vals = [
    results_all[(2, "shake_ocsort")]["lock_rate"],
    results_all[(2, "shake_stab_ocsort")]["lock_rate"],
    results_all[(10, "shake_ocsort")]["lock_rate"],
    results_all[(10, "shake_stab_ocsort")]["lock_rate"],
]

bot_vals = [
    results_bot[(2, "shake_botsort")]["lock_rate"],
    results_bot[(2, "shake_stab_botsort")]["lock_rate"],
    results_bot[(10, "shake_botsort")]["lock_rate"],
    results_bot[(10, "shake_stab_botsort")]["lock_rate"],
]

x = np.arange(len(labels))
width = 0.35

ax.bar(x - width/2, oc_vals, width, label="OC-SORT")
ax.bar(x + width/2, bot_vals, width, label="BoT-SORT")

ax.set_title("Tracker sensitivity check: lock rate under shaken conditions")
ax.set_ylabel("Lock rate")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{EXPORT_DIR}/fig_tracker_sensitivity_lock_rate.png", dpi=300, bbox_inches="tight")
plt.show()

print("saved:", f"{EXPORT_DIR}/fig_tracker_sensitivity_lock_rate.png")


In [ ]:
print("=== KEY NUMBERS TO REMEMBER ===")

for tid in TARGET_TIDS:
    o = results_all[(tid, "orig_ocsort")]
    os_ = results_all[(tid, "orig_stab_ocsort")]
    s = results_all[(tid, "shake_ocsort")]
    ss = results_all[(tid, "shake_stab_ocsort")]

    print(f"\nTarget {tid}:")
    print(f"  Original -> Original+V2: lock_rate {o['lock_rate']:.4f} -> {os_['lock_rate']:.4f}")
    print(f"  Original -> Original+V2: lost_segments {o['lost_segments']} -> {os_['lost_segments']}")
    print(f"  Original -> Original+V2: longest_run {o['longest_continuous_run']} -> {os_['longest_continuous_run']}")
    print(f"  Original -> Original+V2: id_switches {o['id_switches']} -> {os_['id_switches']}")

    print(f"  Shaken -> Shaken+V2: lock_rate {s['lock_rate']:.4f} -> {ss['lock_rate']:.4f}")
    print(f"  Shaken -> Shaken+V2: lost_segments {s['lost_segments']} -> {ss['lost_segments']}")
    print(f"  Shaken -> Shaken+V2: longest_run {s['longest_continuous_run']} -> {ss['longest_continuous_run']}")
    print(f"  Shaken -> Shaken+V2: id_switches {s['id_switches']} -> {ss['id_switches']}")


Модели CV

In [ ]:
!pip install opencv-python numpy scipy matplotlib -q
print("ok")


In [ ]:
from google.colab import files
import os, shutil

uploaded = files.upload()   # выбери свой simulator_run.mp4
if not uploaded:
    raise SystemExit("no file uploaded")

src_name = next(iter(uploaded))
INPUT_VIDEO = "/content/input.mp4"
shutil.move(src_name, INPUT_VIDEO)
print(f"video at: {INPUT_VIDEO}  ({os.path.getsize(INPUT_VIDEO)/1e6:.1f} MB)")


In [ ]:
import os, cv2, numpy as np

# medfilt с fallback на numpy
try:
    from scipy.signal import medfilt as _scipy_medfilt
    def medfilt(x, kernel_size):
        return _scipy_medfilt(x, kernel_size=kernel_size)
except ImportError:
    def medfilt(x, kernel_size):
        r = kernel_size // 2
        xp = np.pad(x, (r, r), mode='edge')
        out = np.empty_like(x, dtype=np.float64)
        for i in range(len(x)):
            out[i] = np.median(xp[i:i+kernel_size])
        return out

# параметры
FEATURE_PARAMS = dict(maxCorners=500, qualityLevel=0.005,
                      minDistance=12, blockSize=3)
LK_PARAMS = dict(winSize=(21, 21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
FB_ERR_THRESHOLD = 1.0   # forward-backward round-trip LK error, px

# блоки пайплайна
def detect_features(gray):
    return cv2.goodFeaturesToTrack(gray, mask=None, **FEATURE_PARAMS)

def track_features_fb(prev_gray, curr_gray, prev_pts):
    """LK + forward-backward error check. Плохие треки отбрасываются."""
    if prev_pts is None or len(prev_pts) == 0:
        return None, None
    curr_pts, st_f, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None, **LK_PARAMS)
    if curr_pts is None or st_f is None:
        return None, None
    prev_back, st_b, _ = cv2.calcOpticalFlowPyrLK(curr_gray, prev_gray, curr_pts, None, **LK_PARAMS)
    if prev_back is None or st_b is None:
        return None, None
    good = (st_f.flatten() == 1) & (st_b.flatten() == 1)
    err = np.linalg.norm(prev_back.reshape(-1, 2) - prev_pts.reshape(-1, 2), axis=1)
    good &= (err < FB_ERR_THRESHOLD)
    if not np.any(good):
        return None, None
    return prev_pts[good], curr_pts[good]

def estimate_affine(prev_pts, curr_pts):
    if prev_pts is None or curr_pts is None or len(prev_pts) < 3:
        return None
    M, _ = cv2.estimateAffinePartial2D(
        prev_pts, curr_pts, method=cv2.RANSAC,
        ransacReprojThreshold=2.0, maxIters=3000,
        confidence=0.999, refineIters=15)
    return M

def decompose_transform(M):
    if M is None: return 0.0, 0.0, 0.0
    return float(M[0, 2]), float(M[1, 2]), float(np.arctan2(M[1, 0], M[0, 0]))

def build_transform(dx, dy, da):
    return np.array([[np.cos(da), -np.sin(da), dx],
                     [np.sin(da),  np.cos(da), dy]], dtype=np.float64)

def robust_std(x):
    """MAD-based std — не чувствителен к выбросам."""
    mad = np.median(np.abs(x - np.median(x)))
    return mad / 0.6745

def clean_transforms(transforms, median_kernel=5, mad_k=4.0, verbose=True):
    """medfilt убирает одиночные спайки, MAD-clip — страховка по хвостам."""
    T = transforms.copy()
    for i in range(3):
        T[:, i] = medfilt(T[:, i], kernel_size=median_kernel)
    for i, name in enumerate(['dx', 'dy', 'da']):
        rstd = robust_std(T[:, i])
        thr = max(mad_k * rstd, 1e-6)
        n_clip = int(np.sum(np.abs(T[:, i]) > thr))
        T[:, i] = np.clip(T[:, i], -thr, thr)
        if verbose:
            print(f"  [clean] {name}: robust_std={rstd:.4f}  thr={thr:.4f}  clipped={n_clip}")
    return T

def moving_average(curve, radius):
    w = 2 * radius + 1
    kernel = np.ones(w) / w
    padded = np.pad(curve, (radius, radius), mode='reflect')
    smoothed = np.convolve(padded, kernel, mode='same')
    return smoothed[radius:-radius]

def smooth_trajectory(trajectory, radius=60):
    out = np.zeros_like(trajectory, dtype=np.float64)
    for i in range(trajectory.shape[1]):
        out[:, i] = moving_average(trajectory[:, i], radius)
    return out

# основной вызов
def stabilize(input_path, output_path,
              side_by_side_path=None,
              transforms_path=None, raw_transforms_path=None,
              trajectory_path=None, smoothed_trajectory_path=None,
              smoothing_radius=None, min_points=20,
              median_kernel=5, mad_k=4.0, verbose=True):
    assert os.path.exists(input_path), input_path

    cap = cv2.VideoCapture(input_path)
    assert cap.isOpened(), f"cannot open {input_path}"
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if smoothing_radius is None:
        smoothing_radius = max(15, int(round(2 * fps)))

    if verbose:
        print(f"[info] {w}x{h} @ {fps:.2f} fps, {n} frames")
        print(f"[info] smoothing_radius = {smoothing_radius} ({(2*smoothing_radius+1)/fps:.2f}s)")

    # per-frame transforms
    ok, prev = cap.read(); assert ok
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

    transforms, last_valid = [], np.zeros(3)
    cnt = dict(detect_fail=0, track_fail=0, affine_fail=0, ok=0)

    for i in range(n - 1):
        prev_pts = detect_features(prev_gray)
        ok, curr = cap.read()
        if not ok: break
        curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)

        if prev_pts is None or len(prev_pts) < min_points:
            cnt['detect_fail'] += 1
            transforms.append(last_valid.copy()); prev_gray = curr_gray; continue

        prev_good, curr_good = track_features_fb(prev_gray, curr_gray, prev_pts)
        if prev_good is None or len(prev_good) < min_points:
            cnt['track_fail'] += 1
            transforms.append(last_valid.copy()); prev_gray = curr_gray; continue

        M = estimate_affine(prev_good, curr_good)
        if M is None:
            cnt['affine_fail'] += 1
            transforms.append(last_valid.copy()); prev_gray = curr_gray; continue

        dx, dy, da = decompose_transform(M)
        last_valid = np.array([dx, dy, da]); transforms.append(last_valid.copy())
        cnt['ok'] += 1; prev_gray = curr_gray

        if verbose and (i + 1) % 200 == 0:
            print(f"  [pass 1] {i+1}/{n-1}  survived_tracks={len(prev_good)}")

    transforms_raw = np.asarray(transforms, dtype=np.float64)
    if verbose: print(f"[pass 1] done: {cnt}")

    # cleanup
    if verbose: print("[clean] medfilt + MAD clip:")
    transforms = clean_transforms(transforms_raw, median_kernel, mad_k, verbose)

    # trajectory + smoothing
    trajectory = np.cumsum(transforms, axis=0)
    smoothed   = smooth_trajectory(trajectory, radius=smoothing_radius)
    transforms_smooth = transforms + (smoothed - trajectory)

    # PASS 2
    cap.release(); cap = cv2.VideoCapture(input_path)
    os.makedirs(os.path.dirname(os.path.abspath(output_path)) or ".", exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    sbs = None
    if side_by_side_path:
        os.makedirs(os.path.dirname(os.path.abspath(side_by_side_path)) or ".", exist_ok=True)
        sbs = cv2.VideoWriter(side_by_side_path, fourcc, fps, (2 * w, h))

    ok, frame = cap.read(); writer.write(frame)
    if sbs is not None: sbs.write(np.hstack([frame, frame]))

    for i in range(n - 1):
        ok, frame = cap.read()
        if not ok or i >= len(transforms_smooth): break
        dx, dy, da = transforms_smooth[i]
        M = build_transform(dx, dy, da)
        stab = cv2.warpAffine(frame, M, (w, h),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REPLICATE)
        writer.write(stab)
        if sbs is not None: sbs.write(np.hstack([frame, stab]))

    cap.release(); writer.release()
    if sbs is not None: sbs.release()

    if transforms_path:         np.save(transforms_path, transforms)
    if raw_transforms_path:     np.save(raw_transforms_path, transforms_raw)
    if trajectory_path:         np.save(trajectory_path, trajectory)
    if smoothed_trajectory_path: np.save(smoothed_trajectory_path, smoothed)

    if verbose:
        print("[done]")
        print(f"  stabilized   -> {output_path}")
        if side_by_side_path: print(f"  side-by-side -> {side_by_side_path}")

    return dict(transforms=transforms, transforms_raw=transforms_raw,
                trajectory=trajectory, smoothed_trajectory=smoothed,
                counters=cnt,
                meta=dict(width=w, height=h, fps=fps, n_frames=n,
                          smoothing_radius=smoothing_radius))

print("stabilize() ready")


In [ ]:
OUTPUT_DIR = "/content/results_v1_tuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = stabilize(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR}/v1_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR}/v1_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR}/v1_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR}/v1_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR}/v1_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR}/v1_smoothed.npy",
    smoothing_radius        = None,   # auto = 2 * fps
    min_points              = 20,
    median_kernel           = 5,
    mad_k                   = 4.0,
    verbose                 = True,
)
print("counters:", results["counters"])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

raw   = results["transforms_raw"]
clean = results["transforms"]
traj  = results["trajectory"]
sm    = results["smoothed_trajectory"]

# per-frame: до vs после очистки
labels_pf = ["dx per frame (px)", "dy per frame (px)", "da per frame (rad)"]
fig1, axes1 = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for i, ax in enumerate(axes1):
    ax.plot(raw[:, i],   color="tab:blue",   alpha=0.35, linewidth=0.7, label="raw")
    ax.plot(clean[:, i], color="tab:orange", linewidth=1.0, label="cleaned")
    ax.axhline(0, color="red", linewidth=0.5)
    ax.set_ylabel(labels_pf[i]); ax.grid(alpha=0.3); ax.legend(loc="upper right")
axes1[-1].set_xlabel("frame index")
fig1.suptitle("Variant 1 (tuned) - per-frame transforms: raw vs cleaned")
fig1.tight_layout()
fig1.savefig(f"{OUTPUT_DIR}/v1_transforms_cmp.png", dpi=120)
plt.show()

# кумулятивная траектория
labels_tr = ["dx cum (px)", "dy cum (px)", "da cum (rad)"]
fig2, axes2 = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for i, ax in enumerate(axes2):
    ax.plot(traj[:, i], label="raw trajectory", alpha=0.7)
    ax.plot(sm[:, i],   label="smoothed", linewidth=2)
    ax.set_ylabel(labels_tr[i]); ax.grid(alpha=0.3); ax.legend(loc="upper right")
axes2[-1].set_xlabel("frame index")
fig2.suptitle("Variant 1 (tuned) - trajectory vs smoothed")
fig2.tight_layout()
fig2.savefig(f"{OUTPUT_DIR}/v1_trajectory.png", dpi=120)
plt.show()

for i, name in enumerate(['dx','dy','da']):
    r = raw[:, i]; c = clean[:, i]
    print(f"{name}: raw std={r.std():.3f} max|.|={np.max(np.abs(r)):.3f}   "
          f"cleaned std={c.std():.3f} max|.|={np.max(np.abs(c)):.3f}")


In [ ]:
from google.colab import files
for f in ["v1_stabilized.mp4", "v1_side_by_side.mp4",
          "v1_transforms_clean.npy", "v1_transforms_raw.npy",
          "v1_trajectory.npy", "v1_smoothed.npy",
          "v1_transforms_cmp.png", "v1_trajectory.png"]:
    path = f"{OUTPUT_DIR}/{f}"
    if os.path.exists(path):
        files.download(path)


VARIANT 2

In [ ]:
import numpy as np

def kalman_smooth_axis(z, q=1e-3, r=1.0, mode='rts'):
    """1D Kalman для константно-скоростной модели."""
    n = len(z)
    F = np.array([[1.0, 1.0], [0.0, 1.0]])
    Q = q * np.array([[0.25, 0.5], [0.5, 1.0]])

    x_pred  = np.zeros((n, 2)); P_pred = np.zeros((n, 2, 2))
    x_filt  = np.zeros((n, 2)); P_filt = np.zeros((n, 2, 2))

    x_filt[0] = [z[0], 0.0]
    P_filt[0] = np.eye(2) * 10.0

    for k in range(1, n):
        x_pred[k] = F @ x_filt[k-1]
        P_pred[k] = F @ P_filt[k-1] @ F.T + Q
        S = P_pred[k][0, 0] + r
        K = P_pred[k][:, 0] / S                       # gain
        x_filt[k] = x_pred[k] + K * (z[k] - x_pred[k][0])
        P_filt[k] = P_pred[k] - np.outer(K, P_pred[k][0, :])

    if mode == 'filter':
        return x_filt[:, 0]

    x_smooth = np.zeros((n, 2)); P_smooth = np.zeros((n, 2, 2))
    x_smooth[-1] = x_filt[-1]; P_smooth[-1] = P_filt[-1]
    for k in range(n - 2, -1, -1):
        try:
            C = P_filt[k] @ F.T @ np.linalg.inv(P_pred[k+1])
        except np.linalg.LinAlgError:
            C = np.zeros((2, 2))
        x_smooth[k] = x_filt[k] + C @ (x_smooth[k+1] - x_pred[k+1])
        P_smooth[k] = P_filt[k] + C @ (P_smooth[k+1] - P_pred[k+1]) @ C.T
    return x_smooth[:, 0]

def kalman_smooth_trajectory(trajectory, q=(1e-3, 1e-3, 1e-5), r=(1.0, 1.0, 1e-3),
                             mode='rts'):
    """Применяем Kalman поканально к (dx_cum, dy_cum, da_cum)."""
    out = np.zeros_like(trajectory, dtype=np.float64)
    for i in range(3):
        out[:, i] = kalman_smooth_axis(trajectory[:, i],
                                       q=q[i], r=r[i], mode=mode)
    return out

def stabilize_v2(input_path, output_path,
                 side_by_side_path=None,
                 transforms_path=None, raw_transforms_path=None,
                 trajectory_path=None, smoothed_trajectory_path=None,
                 q=(1e-3, 1e-3, 1e-5), r=(1.0, 1.0, 1e-3), kalman_mode='rts',
                 min_points=20, median_kernel=5, mad_k=4.0, verbose=True):
    """
    То же самое, что stabilize() из Ячейки 3, но smooth_trajectory заменён на Kalman.
    Все остальные блоки (FB-LK, RANSAC, medfilt+clip, warpAffine) идентичны V1.
    """
    import os, cv2, numpy as np
    assert os.path.exists(input_path)

    cap = cv2.VideoCapture(input_path)
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if verbose:
        print(f"[V2] {w}x{h} @ {fps:.2f} fps, {n} frames; kalman_mode={kalman_mode}")
        print(f"[V2] q={q}  r={r}")

    ok, prev = cap.read()
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    transforms, last_valid = [], np.zeros(3)
    cnt = dict(detect_fail=0, track_fail=0, affine_fail=0, ok=0)

    for i in range(n - 1):
        prev_pts = detect_features(prev_gray)
        ok, curr = cap.read()
        if not ok: break
        curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)

        if prev_pts is None or len(prev_pts) < min_points:
            cnt['detect_fail'] += 1; transforms.append(last_valid.copy())
            prev_gray = curr_gray; continue

        prev_good, curr_good = track_features_fb(prev_gray, curr_gray, prev_pts)
        if prev_good is None or len(prev_good) < min_points:
            cnt['track_fail'] += 1; transforms.append(last_valid.copy())
            prev_gray = curr_gray; continue

        M = estimate_affine(prev_good, curr_good)
        if M is None:
            cnt['affine_fail'] += 1; transforms.append(last_valid.copy())
            prev_gray = curr_gray; continue

        dx, dy, da = decompose_transform(M)
        last_valid = np.array([dx, dy, da])
        transforms.append(last_valid.copy())
        cnt['ok'] += 1; prev_gray = curr_gray

    transforms_raw = np.asarray(transforms, dtype=np.float64)
    if verbose: print(f"[V2 pass 1] done: {cnt}")

    transforms = clean_transforms(transforms_raw, median_kernel, mad_k, verbose)

    trajectory = np.cumsum(transforms, axis=0)
    smoothed   = kalman_smooth_trajectory(trajectory, q=q, r=r, mode=kalman_mode)
    transforms_smooth = transforms + (smoothed - trajectory)

    cap.release(); cap = cv2.VideoCapture(input_path)
    os.makedirs(os.path.dirname(os.path.abspath(output_path)) or ".", exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    sbs = None
    if side_by_side_path:
        sbs = cv2.VideoWriter(side_by_side_path, fourcc, fps, (2 * w, h))

    ok, frame = cap.read(); writer.write(frame)
    if sbs is not None: sbs.write(np.hstack([frame, frame]))

    for i in range(n - 1):
        ok, frame = cap.read()
        if not ok or i >= len(transforms_smooth): break
        dx, dy, da = transforms_smooth[i]
        M = build_transform(dx, dy, da)
        stab = cv2.warpAffine(frame, M, (w, h),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REPLICATE)
        writer.write(stab)
        if sbs is not None: sbs.write(np.hstack([frame, stab]))

    cap.release(); writer.release()
    if sbs is not None: sbs.release()

    if transforms_path:         np.save(transforms_path, transforms)
    if raw_transforms_path:     np.save(raw_transforms_path, transforms_raw)
    if trajectory_path:         np.save(trajectory_path, trajectory)
    if smoothed_trajectory_path: np.save(smoothed_trajectory_path, smoothed)

    if verbose:
        print(f"[V2 done] stabilized -> {output_path}")
        if side_by_side_path: print(f"          side-by-side -> {side_by_side_path}")

    return dict(transforms=transforms, transforms_raw=transforms_raw,
                trajectory=trajectory, smoothed_trajectory=smoothed,
                counters=cnt,
                meta=dict(width=w, height=h, fps=fps, n_frames=n, q=q, r=r))

print("kalman_smooth_trajectory + stabilize_v2 ready")


In [ ]:
import os
OUTPUT_DIR_V2 = "/content/results_v2"
os.makedirs(OUTPUT_DIR_V2, exist_ok=True)

results_v2 = stabilize_v2(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR_V2}/v2_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR_V2}/v2_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR_V2}/v2_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR_V2}/v2_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR_V2}/v2_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR_V2}/v2_smoothed.npy",
    q=(1e-3, 1e-3, 1e-5),    # process noise (меньше → плавнее, но больше лаг)
    r=(1.0, 1.0, 1e-3),       # measurement noise (больше → больше доверяем модели)
    kalman_mode='rts',        # 'rts' — RTS smoother (оффлайн, аналог MA)
                              # 'filter' — каузально, как online
    min_points=20, median_kernel=5, mad_k=4.0, verbose=True,
)
print("counters:", results_v2["counters"])


V2 B_R100 не берем как рабочий кандидат.

слишком агрессивно сглаживает,
начинает искажать геометрию сцены,
и для warehouse environment это плохо, потому что там как раз много жестких вертикальных/горизонтальных структур.

Оставляем:
V1 (LK + MA) — baseline
V2 A_R10 — tuned Kalman variant

In [ ]:
# метрики

import os, time, numpy as np, cv2

def _reestimate_transforms(video_path, sample_stride=1, verbose=False):
    """Повторно прогоняем тот же простой LK-пайплайн на УЖЕ стабилизированном видео."""
    cap = cv2.VideoCapture(video_path)
    ok, prev = cap.read()
    if not ok: return np.zeros((0, 3))
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    out = []
    while True:
        ok, curr = cap.read()
        if not ok: break
        curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)
        pts = cv2.goodFeaturesToTrack(prev_gray, mask=None,
                                       maxCorners=300, qualityLevel=0.01,
                                       minDistance=15, blockSize=3)
        if pts is None or len(pts) < 10:
            out.append([0.0, 0.0, 0.0]); prev_gray = curr_gray; continue
        new_pts, st, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, pts, None,
                                                   winSize=(21, 21), maxLevel=3)
        if new_pts is None:
            out.append([0.0, 0.0, 0.0]); prev_gray = curr_gray; continue
        st = st.flatten().astype(bool)
        if st.sum() < 10:
            out.append([0.0, 0.0, 0.0]); prev_gray = curr_gray; continue
        M, _ = cv2.estimateAffinePartial2D(pts[st], new_pts[st], method=cv2.RANSAC)
        if M is None:
            out.append([0.0, 0.0, 0.0])
        else:
            out.append([float(M[0, 2]), float(M[1, 2]),
                        float(np.arctan2(M[1, 0], M[0, 0]))])
        prev_gray = curr_gray
    cap.release()
    return np.asarray(out, dtype=np.float64)

def _itf_psnr(video_path):
    """Inter-frame Transformation Fidelity: средний PSNR между соседними кадрами."""
    cap = cv2.VideoCapture(video_path)
    ok, prev = cap.read()
    if not ok: return float('nan')
    psnrs = []
    while True:
        ok, curr = cap.read()
        if not ok: break
        psnrs.append(cv2.PSNR(prev, curr))
        prev = curr
    cap.release()
    return float(np.mean(psnrs)) if psnrs else float('nan')

def _stability_score(trajectory, k_low=5):
    """
    Liu et al. 2013 stability: доля энергии в k_low низших ненулевых частотах.
    Чем выше — тем более «гладкая» траектория.
    """scores = []."""
    Доля валидных пикселей после warpAffine (метрика «потери поля зрения»).
    Считаем по выборке кадров для скорости.
    """if len(transforms_smooth) == 0: return float('nan')."""
    kwargs = dict(kwargs)
    kwargs['verbose'] = False
    t0 = time.time()
    res = stabilize_callable(*args, **kwargs)
    elapsed = time.time() - t0
    n = res['meta']['n_frames']
    return 1000.0 * elapsed / max(n, 1)

def compute_metrics(name, original_video, stabilized_video,
                    raw_transforms, smoothed_trajectory):
    """
    Вычисляет 5 метрик и возвращает dict.
    """# 1."""
    keys = list(rows[0].keys())
    widths = {k: max(len(k), max(len(str(r[k])) for r in rows)) for k in keys}
    line = " | ".join(f"{k:>{widths[k]}}" for k in keys); print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(f"{str(r[k]):>{widths[k]}}" for k in keys))

print("metrics module ready")


In [ ]:
import numpy as np

V1_DIR = "/content/results_v1_tuned"
V2_DIR = "/content/results_v2"

m1 = compute_metrics(
    name="V1 (LK + MA)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V1_DIR}/v1_stabilized.mp4",
    raw_transforms=np.load(f"{V1_DIR}/v1_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V1_DIR}/v1_smoothed.npy"),
)
m2 = compute_metrics(
    name="V2 (LK + Kalman)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V2_DIR}/v2_stabilized.mp4",
    raw_transforms=np.load(f"{V2_DIR}/v2_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V2_DIR}/v2_smoothed.npy"),
)

print_comparison([m1, m2])

# и в виде CSV для вставки в отчёт / Excel:
import csv, io
buf = io.StringIO()
w = csv.DictWriter(buf, fieldnames=list(m1.keys()))
w.writeheader(); w.writerow(m1); w.writerow(m2)
print("\nCSV:\n" + buf.getvalue())


VARIANT 3

In [ ]:
import cv2, numpy as np, os

def stabilize_v3(input_path, output_path,
                 side_by_side_path=None,
                 transforms_path=None, raw_transforms_path=None,
                 trajectory_path=None, smoothed_trajectory_path=None,
                 nfeatures=2500, ratio=0.70, min_matches=30,
                 smoothing_radius=None,
                 median_kernel=5, mad_k=4.0, verbose=True):
    """Variant 3 (tuned): ORB + BFMatcher (Hamming, ratio test)."""
    assert os.path.exists(input_path)

    cap = cv2.VideoCapture(input_path)
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if smoothing_radius is None:
        smoothing_radius = max(15, int(round(2 * fps)))

    if verbose:
        print(f"[V3] {w}x{h} @ {fps:.2f} fps, {n} frames")
        print(f"[V3] nfeatures={nfeatures}  ratio={ratio}  "
              f"min_matches={min_matches}  smoothing_radius={smoothing_radius}")

    orb = cv2.ORB_create(
        nfeatures=nfeatures, scaleFactor=1.2, nlevels=8,
        edgeThreshold=15, WTA_K=2,
        scoreType=cv2.ORB_HARRIS_SCORE,
        patchSize=31, fastThreshold=10,
    )
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

    ok, prev = cap.read(); assert ok
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    prev_kp, prev_des = orb.detectAndCompute(prev_gray, None)

    transforms, last_valid = [], np.zeros(3)
    cnt = dict(detect_fail=0, match_fail=0, affine_fail=0, ok=0)

    for i in range(n - 1):
        ok, curr = cap.read()
        if not ok: break
        curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)
        curr_kp, curr_des = orb.detectAndCompute(curr_gray, None)

        if (prev_des is None or curr_des is None
                or len(prev_des) < 10 or len(curr_des) < 10):
            cnt['detect_fail'] += 1
            transforms.append(last_valid.copy())
            prev_kp, prev_des = curr_kp, curr_des; continue

        # Lowe's ratio test (стандартная фильтрация ORB-матчей)
        matches = bf.knnMatch(prev_des, curr_des, k=2)
        good = []
        for pair in matches:
            if len(pair) < 2: continue
            m, nbr = pair
            if m.distance < ratio * nbr.distance:
                good.append(m)

        if len(good) < min_matches:
            cnt['match_fail'] += 1
            transforms.append(last_valid.copy())
            prev_kp, prev_des = curr_kp, curr_des; continue

        prev_pts = np.array([prev_kp[m.queryIdx].pt for m in good],
                            dtype=np.float32).reshape(-1, 1, 2)
        curr_pts = np.array([curr_kp[m.trainIdx].pt for m in good],
                            dtype=np.float32).reshape(-1, 1, 2)

        M = estimate_affine(prev_pts, curr_pts)
        if M is None:
            cnt['affine_fail'] += 1
            transforms.append(last_valid.copy())
            prev_kp, prev_des = curr_kp, curr_des; continue

        dx, dy, da = decompose_transform(M)
        last_valid = np.array([dx, dy, da])
        transforms.append(last_valid.copy())
        cnt['ok'] += 1
        prev_kp, prev_des = curr_kp, curr_des

        if verbose and (i + 1) % 200 == 0:
            print(f"  [V3 pass 1] {i+1}/{n-1}  good_matches={len(good)}")

    transforms_raw = np.asarray(transforms, dtype=np.float64)
    if verbose: print(f"[V3 pass 1] done: {cnt}")

    transforms = clean_transforms(transforms_raw, median_kernel, mad_k, verbose)
    trajectory = np.cumsum(transforms, axis=0)
    smoothed   = smooth_trajectory(trajectory, radius=smoothing_radius)
    transforms_smooth = transforms + (smoothed - trajectory)

    cap.release(); cap = cv2.VideoCapture(input_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    sbs = None
    if side_by_side_path:
        os.makedirs(os.path.dirname(os.path.abspath(side_by_side_path)) or ".",
                    exist_ok=True)
        sbs = cv2.VideoWriter(side_by_side_path, fourcc, fps, (2 * w, h))

    ok, frame = cap.read(); writer.write(frame)
    if sbs is not None: sbs.write(np.hstack([frame, frame]))

    for i in range(n - 1):
        ok, frame = cap.read()
        if not ok or i >= len(transforms_smooth): break
        dx, dy, da = transforms_smooth[i]
        M = build_transform(dx, dy, da)
        stab = cv2.warpAffine(frame, M, (w, h),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REPLICATE)
        writer.write(stab)
        if sbs is not None: sbs.write(np.hstack([frame, stab]))

    cap.release(); writer.release()
    if sbs is not None: sbs.release()

    if transforms_path:         np.save(transforms_path, transforms)
    if raw_transforms_path:     np.save(raw_transforms_path, transforms_raw)
    if trajectory_path:         np.save(trajectory_path, trajectory)
    if smoothed_trajectory_path: np.save(smoothed_trajectory_path, smoothed)

    if verbose: print(f"[V3 done] -> {output_path}")

    return dict(transforms=transforms, transforms_raw=transforms_raw,
                trajectory=trajectory, smoothed_trajectory=smoothed,
                counters=cnt,
                meta=dict(width=w, height=h, fps=fps, n_frames=n,
                          smoothing_radius=smoothing_radius))

print("stabilize_v3 (tuned) ready")


In [ ]:
import os, time

OUTPUT_DIR_V3 = "/content/results_v3"
os.makedirs(OUTPUT_DIR_V3, exist_ok=True)

t0 = time.time()
results_v3 = stabilize_v3(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR_V3}/v3_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR_V3}/v3_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR_V3}/v3_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR_V3}/v3_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR_V3}/v3_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR_V3}/v3_smoothed.npy",
    nfeatures=2500,        # тюнингованное
    ratio=0.70,            # тюнингованное
    min_matches=30,        # тюнингованное
    smoothing_radius=None, # auto = 2*fps, как в V1
    median_kernel=5, mad_k=4.0,
    verbose=True,
)

dt = time.time() - t0
n  = results_v3['meta']['n_frames']
print(f"\n[V3] elapsed: {dt:.1f}s  ({1000*dt/n:.1f} ms/frame, "
      f"~{n/dt:.1f} fps effective)")
print("counters:", results_v3["counters"])


In [ ]:
import numpy as np

V1_DIR    = "/content/results_v1_tuned"
V2_DIR    = "/content/results_v2_A_R10"
V3_DIR    = "/content/results_v3"

m_v1 = compute_metrics(
    name="V1 (LK + MA)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V1_DIR}/v1_stabilized.mp4",
    raw_transforms=np.load(f"{V1_DIR}/v1_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V1_DIR}/v1_smoothed.npy"),
)
m_v2 = compute_metrics(
    name="V2 (LK + Kalman, R×10)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V2_DIR}/v2_stabilized.mp4",
    raw_transforms=np.load(f"{V2_DIR}/v2_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V2_DIR}/v2_smoothed.npy"),
)
m_v3 = compute_metrics(
    name="V3 (ORB + MA)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V3_DIR}/v3_stabilized.mp4",
    raw_transforms=np.load(f"{V3_DIR}/v3_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V3_DIR}/v3_smoothed.npy"),
)

print_comparison([m_v1, m_v2, m_v3])


In [ ]:
import time

def quick_speed(func, label, **kwargs):
    out = f"/content/_speed_{label}.mp4"
    kwargs['verbose'] = False
    t0 = time.time()
    res = func(input_path=INPUT_VIDEO, output_path=out, **kwargs)
    dt = time.time() - t0
    n  = res['meta']['n_frames']
    os.remove(out)
    return dict(label=label, total_s=round(dt,2), ms_per_frame=round(1000*dt/n,2),
                effective_fps=round(n/dt,1))

speeds = [
    quick_speed(stabilize,    "V1"),
    quick_speed(stabilize_v2, "V2_A_R10",
                q=(1e-3,1e-3,1e-5), r=(10.0,10.0,1e-2), kalman_mode='rts'),
    quick_speed(stabilize_v3, "V3"),
]
for s in speeds: print(s)


VARIANT 4

In [ ]:
import cv2, numpy as np, os

# Grid-based feature seeding (равномерное покрытие сцены)

def grid_detect(gray, n_target=600, grid=(3, 4), mask=None,
                quality_level=0.005, min_distance=10, block_size=3):
    """Shi-Tomasi на N×M ячейках кадра."""
    h, w = gray.shape
    rows, cols = grid
    cell_h, cell_w = h // rows, w // cols
    per_cell = max(5, int(np.ceil(n_target / (rows * cols))))

    all_pts = []
    for r in range(rows):
        for c in range(cols):
            y0 = r * cell_h; y1 = (r + 1) * cell_h if r < rows - 1 else h
            x0 = c * cell_w; x1 = (c + 1) * cell_w if c < cols - 1 else w
            cell = gray[y0:y1, x0:x1]
            cell_mask = None if mask is None else mask[y0:y1, x0:x1]
            pts = cv2.goodFeaturesToTrack(
                cell, mask=cell_mask, maxCorners=per_cell,
                qualityLevel=quality_level, minDistance=min_distance,
                blockSize=block_size,
            )
            if pts is None or len(pts) == 0: continue
            pts[:, 0, 0] += x0; pts[:, 0, 1] += y0
            all_pts.append(pts)

    if not all_pts: return None
    return np.vstack(all_pts).astype(np.float32)

def make_avoidance_mask(shape, pts, radius=12):
    """Маска, исключающая окрестности уже существующих keypoint'ов."""
    mask = np.full(shape, 255, dtype=np.uint8)
    if pts is not None and len(pts) > 0:
        for pt in pts.reshape(-1, 2):
            cv2.circle(mask, (int(pt[0]), int(pt[1])), radius, 0, -1)
    return mask

# Variant 4 — robust LK pipeline

def stabilize_v4(input_path, output_path,
                 side_by_side_path=None,
                 transforms_path=None, raw_transforms_path=None,
                 trajectory_path=None, smoothed_trajectory_path=None,
                 n_target=600, n_min=80, grid=(3, 4),
                 affine_resid_thr=3.0,
                 smoothing='ma',                       # 'ma' или 'kalman'
                 smoothing_radius=None,                # для MA
                 kalman_q=(1e-3, 1e-3, 1e-5),          # для Kalman
                 kalman_r=(10.0, 10.0, 1e-2),          # настройка V2 A_R10
                 median_kernel=5, mad_k=4.0,
                 verbose=True):
    """Variant 4: robust LK с track-persistence и фильтрацией плохих треков."""
    assert os.path.exists(input_path)

    cap = cv2.VideoCapture(input_path)
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if smoothing_radius is None:
        smoothing_radius = max(15, int(round(2 * fps)))

    if verbose:
        print(f"[V4] {w}x{h} @ {fps:.2f} fps, {n} frames")
        print(f"[V4] grid={grid}  n_target={n_target}  n_min={n_min}  "
              f"affine_resid_thr={affine_resid_thr}  smoothing={smoothing}")

    ok, prev = cap.read(); assert ok
    prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    prev_pts  = grid_detect(prev_gray, n_target=n_target, grid=grid)

    transforms, last_valid = [], np.zeros(3)
    cnt = dict(detect_fail=0, track_fail=0, affine_fail=0,
               reseed=0, top_up=0, ok=0)
    track_age = 0

    for i in range(n - 1):
        ok, curr = cap.read()
        if not ok: break
        curr_gray = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY)

        if prev_pts is None or len(prev_pts) < n_min:
            new_pts = grid_detect(prev_gray, n_target=n_target, grid=grid)
            if new_pts is None or len(new_pts) < n_min:
                cnt['detect_fail'] += 1
                transforms.append(last_valid.copy())
                prev_gray = curr_gray
                continue
            prev_pts = new_pts; cnt['reseed'] += 1; track_age = 0

        prev_good, curr_good = track_features_fb(prev_gray, curr_gray, prev_pts)
        if prev_good is None or len(prev_good) < n_min:
            cnt['track_fail'] += 1
            transforms.append(last_valid.copy())
            prev_pts = None    # форсим re-seed на следующей итерации
            prev_gray = curr_gray
            continue

        M, inliers = cv2.estimateAffinePartial2D(
            prev_good, curr_good, method=cv2.RANSAC,
            ransacReprojThreshold=2.0, maxIters=3000,
            confidence=0.999, refineIters=15)
        if M is None:
            cnt['affine_fail'] += 1
            transforms.append(last_valid.copy())
            prev_pts = None; prev_gray = curr_gray
            continue
        if inliers is not None:
            mask_in = inliers.flatten().astype(bool)
            prev_good = prev_good[mask_in]; curr_good = curr_good[mask_in]

        if len(prev_good) > 3:
            pp = prev_good.reshape(-1, 2)
            pp_h = np.hstack([pp, np.ones((len(pp), 1))])
            predicted = pp_h @ M.T
            actual    = curr_good.reshape(-1, 2)
            resid = np.linalg.norm(predicted - actual, axis=1)
            keep = resid < affine_resid_thr
            prev_good = prev_good[keep]; curr_good = curr_good[keep]

        dx, dy, da = decompose_transform(M)
        last_valid = np.array([dx, dy, da])
        transforms.append(last_valid.copy())
        cnt['ok'] += 1; track_age += 1

        prev_pts = curr_good.reshape(-1, 1, 2) if len(curr_good) > 0 else None
        prev_gray = curr_gray

        if prev_pts is not None and len(prev_pts) < n_target // 2:
            avoid = make_avoidance_mask(curr_gray.shape, prev_pts, radius=12)
            extra_n = n_target - len(prev_pts)
            extra = grid_detect(curr_gray, n_target=extra_n,
                                grid=grid, mask=avoid)
            if extra is not None and len(extra) > 0:
                prev_pts = np.vstack([prev_pts, extra])
                cnt['top_up'] += 1

        if verbose and (i + 1) % 200 == 0:
            print(f"  [V4 pass 1] {i+1}/{n-1}  tracks={len(prev_good)}  "
                  f"age={track_age}  reseed={cnt['reseed']}  top_up={cnt['top_up']}")

    transforms_raw = np.asarray(transforms, dtype=np.float64)
    if verbose: print(f"[V4 pass 1] done: {cnt}")

    transforms = clean_transforms(transforms_raw, median_kernel, mad_k, verbose)
    trajectory = np.cumsum(transforms, axis=0)

    if smoothing == 'kalman':
        smoothed = kalman_smooth_trajectory(trajectory,
                                            q=kalman_q, r=kalman_r, mode='rts')
    else:
        smoothed = smooth_trajectory(trajectory, radius=smoothing_radius)

    transforms_smooth = transforms + (smoothed - trajectory)

    cap.release(); cap = cv2.VideoCapture(input_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    sbs = None
    if side_by_side_path:
        os.makedirs(os.path.dirname(os.path.abspath(side_by_side_path)) or ".",
                    exist_ok=True)
        sbs = cv2.VideoWriter(side_by_side_path, fourcc, fps, (2 * w, h))

    ok, frame = cap.read(); writer.write(frame)
    if sbs is not None: sbs.write(np.hstack([frame, frame]))

    for i in range(n - 1):
        ok, frame = cap.read()
        if not ok or i >= len(transforms_smooth): break
        dx, dy, da = transforms_smooth[i]
        M = build_transform(dx, dy, da)
        stab = cv2.warpAffine(frame, M, (w, h),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REPLICATE)
        writer.write(stab)
        if sbs is not None: sbs.write(np.hstack([frame, stab]))

    cap.release(); writer.release()
    if sbs is not None: sbs.release()

    if transforms_path:         np.save(transforms_path, transforms)
    if raw_transforms_path:     np.save(raw_transforms_path, transforms_raw)
    if trajectory_path:         np.save(trajectory_path, trajectory)
    if smoothed_trajectory_path: np.save(smoothed_trajectory_path, smoothed)

    if verbose: print(f"[V4 done] -> {output_path}")

    return dict(transforms=transforms, transforms_raw=transforms_raw,
                trajectory=trajectory, smoothed_trajectory=smoothed,
                counters=cnt,
                meta=dict(width=w, height=h, fps=fps, n_frames=n,
                          smoothing_radius=smoothing_radius, smoothing=smoothing))

print("stabilize_v4 ready (with grid_detect / make_avoidance_mask)")


In [ ]:
import os, time

OUTPUT_DIR_V4 = "/content/results_v4"
os.makedirs(OUTPUT_DIR_V4, exist_ok=True)

t0 = time.time()
results_v4 = stabilize_v4(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR_V4}/v4_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR_V4}/v4_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR_V4}/v4_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR_V4}/v4_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR_V4}/v4_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR_V4}/v4_smoothed.npy",
    n_target=600, n_min=80, grid=(3, 4),
    affine_resid_thr=3.0,
    smoothing='ma',
    smoothing_radius=None,
    median_kernel=5, mad_k=4.0,
    verbose=True,
)
print(f"\n[V4]    elapsed: {time.time() - t0:.1f}s")
print(f"        counters: {results_v4['counters']}")

OUTPUT_DIR_V4P = "/content/results_v4_pro"
os.makedirs(OUTPUT_DIR_V4P, exist_ok=True)

t0 = time.time()
results_v4p = stabilize_v4(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR_V4P}/v4_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR_V4P}/v4_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR_V4P}/v4_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR_V4P}/v4_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR_V4P}/v4_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR_V4P}/v4_smoothed.npy",
    n_target=600, n_min=80, grid=(3, 4),
    affine_resid_thr=3.0,
    smoothing='kalman',
    kalman_q=(1e-3, 1e-3, 1e-5),
    kalman_r=(10.0, 10.0, 1e-2),     # точно как в V2 A_R10
    median_kernel=5, mad_k=4.0,
    verbose=True,
)
print(f"\n[V4_pro] elapsed: {time.time() - t0:.1f}s")
print(f"         counters: {results_v4p['counters']}")


прогон V4 (с MA) и V4_pro (с Kalman)

In [ ]:
import os, time

OUTPUT_DIR_V4 = "/content/results_v4"
os.makedirs(OUTPUT_DIR_V4, exist_ok=True)

t0 = time.time()
results_v4 = stabilize_v4(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR_V4}/v4_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR_V4}/v4_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR_V4}/v4_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR_V4}/v4_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR_V4}/v4_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR_V4}/v4_smoothed.npy",
    n_target=600, n_min=80, grid=(3, 4),
    affine_resid_thr=3.0,
    smoothing='ma',
    smoothing_radius=None,
    median_kernel=5, mad_k=4.0,
    verbose=True,
)
print(f"\n[V4]    elapsed: {time.time() - t0:.1f}s")
print(f"        counters: {results_v4['counters']}")

OUTPUT_DIR_V4P = "/content/results_v4_pro"
os.makedirs(OUTPUT_DIR_V4P, exist_ok=True)

t0 = time.time()
results_v4p = stabilize_v4(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR_V4P}/v4_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR_V4P}/v4_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR_V4P}/v4_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR_V4P}/v4_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR_V4P}/v4_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR_V4P}/v4_smoothed.npy",
    n_target=600, n_min=80, grid=(3, 4),
    affine_resid_thr=3.0,
    smoothing='kalman',
    kalman_q=(1e-3, 1e-3, 1e-5),
    kalman_r=(10.0, 10.0, 1e-2),     # точно как в V2 A_R10
    median_kernel=5, mad_k=4.0,
    verbose=True,
)
print(f"\n[V4_pro] elapsed: {time.time() - t0:.1f}s")
print(f"         counters: {results_v4p['counters']}")


In [ ]:
import numpy as np

V1_DIR    = "/content/results_v1_tuned"
V2_DIR    = "/content/results_v2_A_R10"
V3_DIR    = "/content/results_v3"
V4_DIR    = "/content/results_v4"
V4P_DIR   = "/content/results_v4_pro"

m_v1 = compute_metrics(
    name="V1 (LK + MA)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V1_DIR}/v1_stabilized.mp4",
    raw_transforms=np.load(f"{V1_DIR}/v1_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V1_DIR}/v1_smoothed.npy"),
)
m_v2 = compute_metrics(
    name="V2 (LK + Kalman)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V2_DIR}/v2_stabilized.mp4",
    raw_transforms=np.load(f"{V2_DIR}/v2_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V2_DIR}/v2_smoothed.npy"),
)
m_v3 = compute_metrics(
    name="V3 (ORB + MA)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V3_DIR}/v3_stabilized.mp4",
    raw_transforms=np.load(f"{V3_DIR}/v3_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V3_DIR}/v3_smoothed.npy"),
)
m_v4 = compute_metrics(
    name="V4 (Robust LK + MA)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V4_DIR}/v4_stabilized.mp4",
    raw_transforms=np.load(f"{V4_DIR}/v4_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V4_DIR}/v4_smoothed.npy"),
)
m_v4p = compute_metrics(
    name="V4_pro (Robust LK + Kalman)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{V4P_DIR}/v4_stabilized.mp4",
    raw_transforms=np.load(f"{V4P_DIR}/v4_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{V4P_DIR}/v4_smoothed.npy"),
)

print_comparison([m_v1, m_v2, m_v3, m_v4, m_v4p])


In [ ]:
import time, os

def quick_speed(func, label, **kwargs):
    out = f"/content/_speed_{label}.mp4"
    kwargs['verbose'] = False
    t0 = time.time()
    res = func(input_path=INPUT_VIDEO, output_path=out, **kwargs)
    dt = time.time() - t0
    n  = res['meta']['n_frames']
    if os.path.exists(out): os.remove(out)
    return dict(label=label, total_s=round(dt,2),
                ms_per_frame=round(1000*dt/n,2),
                effective_fps=round(n/dt,1))

speeds = [
    quick_speed(stabilize,    "V1"),
    quick_speed(stabilize_v2, "V2",      q=(1e-3,1e-3,1e-5), r=(10.0,10.0,1e-2),
                                          kalman_mode='rts'),
    quick_speed(stabilize_v3, "V3",      nfeatures=2500, ratio=0.70, min_matches=30),
    quick_speed(stabilize_v4, "V4",      smoothing='ma'),
    quick_speed(stabilize_v4, "V4_pro",  smoothing='kalman',
                                          kalman_q=(1e-3,1e-3,1e-5),
                                          kalman_r=(10.0,10.0,1e-2)),
]
for s in speeds: print(s)


сравним 4 про и 2 версии

In [ ]:
import os, numpy as np, cv2, matplotlib.pyplot as plt

V2_DIR  = "/content/results_v2_A_R10"
V4P_DIR = "/content/results_v4_pro"
PLOT_DIR = "/content/comparison_v2_v4pro"
os.makedirs(PLOT_DIR, exist_ok=True)

def _load_set(d, prefix):
    return dict(
        transforms_raw   = np.load(f"{d}/{prefix}_transforms_raw.npy"),
        transforms_clean = np.load(f"{d}/{prefix}_transforms_clean.npy"),
        trajectory       = np.load(f"{d}/{prefix}_trajectory.npy"),
        smoothed         = np.load(f"{d}/{prefix}_smoothed.npy"),
        video            = f"{d}/{prefix}_stabilized.mp4",
    )

v2  = _load_set(V2_DIR,  "v2")
v4p = _load_set(V4P_DIR, "v4")

# fps для частотных графиков
cap = cv2.VideoCapture(INPUT_VIDEO)
FPS = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
cap.release()

C_V2  = "tab:blue"
C_V4P = "tab:red"

print(f"V2  transforms shape: {v2['transforms_clean'].shape}")
print(f"V4p transforms shape: {v4p['transforms_clean'].shape}")
print(f"FPS = {FPS}")


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex='col')
labels_pf  = ["dx per frame (px)", "dy per frame (px)", "da per frame (rad)"]
labels_cum = ["dx cum (px)",       "dy cum (px)",       "da cum (rad)"]

# Left column: per-frame (cleaned) — здесь видно качество feature-блока
for i in range(3):
    axes[i, 0].plot(v2['transforms_clean'][:, i],  color=C_V2,  alpha=0.95,
                    linewidth=0.7, label='V2 (LK + Kalman)')
    axes[i, 0].plot(v4p['transforms_clean'][:, i], color=C_V4P, alpha=0.95,
                    linewidth=0.7, label='V4_pro (Robust LK + Kalman)')
    axes[i, 0].axhline(0, color='black', linewidth=0.3)
    axes[i, 0].set_ylabel(labels_pf[i]); axes[i, 0].grid(alpha=0.3)
    axes[i, 0].legend(loc='upper right', fontsize=9)
axes[0, 0].set_title("Per-frame transforms (после medfilt+clip)")
axes[-1, 0].set_xlabel("frame index")

# Right column: cumulative trajectory + smoothed overlay (по 4 линии на ось)
for i in range(3):
    axes[i, 1].plot(v2['trajectory'][:, i],  color=C_V2,  alpha=0.30,
                    linewidth=0.8, label='V2 raw')
    axes[i, 1].plot(v2['smoothed'][:, i],    color=C_V2,  linewidth=2.0,
                    label='V2 smoothed')
    axes[i, 1].plot(v4p['trajectory'][:, i], color=C_V4P, alpha=0.30,
                    linewidth=0.8, label='V4_pro raw')
    axes[i, 1].plot(v4p['smoothed'][:, i],   color=C_V4P, linewidth=2.0,
                    label='V4_pro smoothed')
    axes[i, 1].set_ylabel(labels_cum[i]); axes[i, 1].grid(alpha=0.3)
    if i == 0:
        axes[i, 1].legend(loc='upper right', fontsize=8, ncol=2)
axes[0, 1].set_title("Кумулятивная траектория (raw + smoothed)")
axes[-1, 1].set_xlabel("frame index")

fig.suptitle("V2 vs V4_pro — motion analysis", fontsize=13)
fig.tight_layout()
fig.savefig(f"{PLOT_DIR}/01_motion_analysis.png", dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axis_labels = ["dx", "dy", "da"]

for i, lab in enumerate(axis_labels):
    s2 = v2['trajectory'][:, i]  - v2['trajectory'][:, i].mean()
    s4 = v4p['trajectory'][:, i] - v4p['trajectory'][:, i].mean()
    sp2 = np.abs(np.fft.rfft(s2))
    sp4 = np.abs(np.fft.rfft(s4))
    freqs = np.fft.rfftfreq(len(s2), d=1.0 / FPS)

    axes[i].loglog(freqs[1:], sp2[1:], color=C_V2,  label='V2',     linewidth=1.2)
    axes[i].loglog(freqs[1:], sp4[1:], color=C_V4P, label='V4_pro', linewidth=1.2)
    axes[i].set_title(f"Spectrum: {lab} (cumulative)")
    axes[i].set_xlabel("frequency (Hz)")
    axes[i].set_ylabel("|FFT|")
    axes[i].grid(True, which='both', alpha=0.3)
    axes[i].legend()

fig.suptitle("Спектр восстановленной траектории — где живёт шум", fontsize=12)
fig.tight_layout()
fig.savefig(f"{PLOT_DIR}/02_spectrum.png", dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
m_v2  = compute_metrics("V2", INPUT_VIDEO, v2['video'],
                        v2['transforms_clean'],  v2['smoothed'])
m_v4p = compute_metrics("V4_pro", INPUT_VIDEO, v4p['video'],
                        v4p['transforms_clean'], v4p['smoothed'])

lower_keys  = ['resid_std_dx', 'resid_std_dy', 'resid_std_da']
higher_keys = ['ITF_gain', 'stability_dx', 'stability_dy', 'stability_da', 'cropping']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

w = 0.36

x = np.arange(len(lower_keys))
v2_vals = [m_v2[k]  for k in lower_keys]
v4_vals = [m_v4p[k] for k in lower_keys]
b1 = ax1.bar(x - w/2, v2_vals, w, color=C_V2,  label='V2')
b2 = ax1.bar(x + w/2, v4_vals, w, color=C_V4P, label='V4_pro')
ax1.set_xticks(x); ax1.set_xticklabels(lower_keys, rotation=10)
ax1.set_title("Residual jitter (↓ ниже = лучше)")
ax1.legend(); ax1.grid(alpha=0.3, axis='y')
for bars, vals in [(b1, v2_vals), (b2, v4_vals)]:
    for b, v in zip(bars, vals):
        ax1.text(b.get_x() + b.get_width()/2, b.get_height(),
                 f"{v:.3g}", ha='center', va='bottom', fontsize=8)

x = np.arange(len(higher_keys))
v2_vals = [m_v2[k]  for k in higher_keys]
v4_vals = [m_v4p[k] for k in higher_keys]
b1 = ax2.bar(x - w/2, v2_vals, w, color=C_V2,  label='V2')
b2 = ax2.bar(x + w/2, v4_vals, w, color=C_V4P, label='V4_pro')
ax2.set_xticks(x); ax2.set_xticklabels(higher_keys, rotation=10)
ax2.set_title("Stability / fidelity (↑ выше = лучше)")
ax2.legend(); ax2.grid(alpha=0.3, axis='y')
for bars, vals in [(b1, v2_vals), (b2, v4_vals)]:
    for b, v in zip(bars, vals):
        ax2.text(b.get_x() + b.get_width()/2, b.get_height(),
                 f"{v:.3g}", ha='center', va='bottom', fontsize=8)

fig.suptitle("Сравнение метрик: V2 vs V4_pro", fontsize=12)
fig.tight_layout()
fig.savefig(f"{PLOT_DIR}/03_metrics_bar.png", dpi=130, bbox_inches='tight')
plt.show()

# Также печатаем разницы — пригодится для отчёта
print("\n=== Differences (V4_pro − V2) ===")
for k in lower_keys + higher_keys:
    diff = m_v4p[k] - m_v2[k]
    direction = "↓" if k in lower_keys else "↑"
    print(f"  {k:18s}: {direction} {diff:+.4f}   (V2={m_v2[k]}, V4_pro={m_v4p[k]})")


In [ ]:
print("Reestimating residual transforms on stabilized videos…")
resid_v2  = _reestimate_transforms(v2['video'])
resid_v4p = _reestimate_transforms(v4p['video'])
print(f"  V2:     n={len(resid_v2)}")
print(f"  V4_pro: n={len(resid_v4p)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axis_labels = [("dx residual (px)",  60),
               ("dy residual (px)",  60),
               ("da residual (rad)", 60)]

for i, (lab, bins) in enumerate(axis_labels):
    s2 = resid_v2[:, i]; s4 = resid_v4p[:, i]
    # общий диапазон, чтобы бины совпадали
    lo, hi = min(s2.min(), s4.min()), max(s2.max(), s4.max())
    edges = np.linspace(lo, hi, bins + 1)

    axes[i].hist(s2, bins=edges, alpha=0.55, color=C_V2,  label='V2')
    axes[i].hist(s4, bins=edges, alpha=0.55, color=C_V4P, label='V4_pro')
    axes[i].axvline(0, color='black', linewidth=0.5)
    axes[i].set_xlabel(lab); axes[i].set_ylabel("count")
    axes[i].grid(alpha=0.3); axes[i].legend()
    axes[i].set_title(f"std V2={s2.std():.3f}   V4_pro={s4.std():.3f}")

fig.suptitle("Остаточный шейк после стабилизации (ближе к 0 = лучше)", fontsize=12)
fig.tight_layout()
fig.savefig(f"{PLOT_DIR}/04_residual_hist.png", dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(15, 11))
gs = GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.28)

# (1) per-frame transforms — берём только dx как репрезентативную ось
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(v2['transforms_clean'][:, 0],  color=C_V2,  linewidth=0.7, label='V2')
ax1.plot(v4p['transforms_clean'][:, 0], color=C_V4P, linewidth=0.7, label='V4_pro')
ax1.axhline(0, color='black', linewidth=0.3)
ax1.set_title("(a) Per-frame dx (cleaned)")
ax1.set_xlabel("frame index"); ax1.set_ylabel("dx (px/frame)")
ax1.grid(alpha=0.3); ax1.legend(fontsize=9)

# (2) cumulative trajectory dx
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(v2['trajectory'][:, 0],  color=C_V2,  alpha=0.3, linewidth=0.7)
ax2.plot(v2['smoothed'][:, 0],    color=C_V2,  linewidth=2.0, label='V2 smoothed')
ax2.plot(v4p['trajectory'][:, 0], color=C_V4P, alpha=0.3, linewidth=0.7)
ax2.plot(v4p['smoothed'][:, 0],   color=C_V4P, linewidth=2.0, label='V4_pro smoothed')
ax2.set_title("(b) Кумулятивная траектория dx")
ax2.set_xlabel("frame index"); ax2.set_ylabel("dx cum (px)")
ax2.grid(alpha=0.3); ax2.legend(fontsize=9)

# (3) спектр dx
ax3 = fig.add_subplot(gs[1, 0])
s2 = v2['trajectory'][:, 0]  - v2['trajectory'][:, 0].mean()
s4 = v4p['trajectory'][:, 0] - v4p['trajectory'][:, 0].mean()
freqs = np.fft.rfftfreq(len(s2), d=1.0 / FPS)
ax3.loglog(freqs[1:], np.abs(np.fft.rfft(s2))[1:], color=C_V2,  label='V2')
ax3.loglog(freqs[1:], np.abs(np.fft.rfft(s4))[1:], color=C_V4P, label='V4_pro')
ax3.set_title("(c) Спектр dx (cumulative)")
ax3.set_xlabel("frequency (Hz)"); ax3.set_ylabel("|FFT|")
ax3.grid(True, which='both', alpha=0.3); ax3.legend(fontsize=9)

# (4) гистограмма остаточного dx
ax4 = fig.add_subplot(gs[1, 1])
s2r = resid_v2[:, 0]; s4r = resid_v4p[:, 0]
lo, hi = min(s2r.min(), s4r.min()), max(s2r.max(), s4r.max())
edges = np.linspace(lo, hi, 50)
ax4.hist(s2r,  bins=edges, alpha=0.55, color=C_V2,  label='V2')
ax4.hist(s4r, bins=edges, alpha=0.55, color=C_V4P, label='V4_pro')
ax4.axvline(0, color='black', linewidth=0.5)
ax4.set_title(f"(d) Остаточный dx после стабилизации\n"
              f"std V2={s2r.std():.2f}  V4_pro={s4r.std():.2f}")
ax4.set_xlabel("residual dx (px)"); ax4.set_ylabel("count")
ax4.grid(alpha=0.3); ax4.legend(fontsize=9)

# (5) Bar chart основных метрик — на всю ширину
ax5 = fig.add_subplot(gs[2, :])
bar_keys = ['resid_std_dx', 'resid_std_dy', 'resid_std_da',
            'cropping', 'stability_dx', 'stability_dy', 'stability_da']
x = np.arange(len(bar_keys))
v2_vals = [m_v2[k]  for k in bar_keys]
v4_vals = [m_v4p[k] for k in bar_keys]
ax5.bar(x - 0.18, v2_vals, 0.36, color=C_V2,  label='V2')
ax5.bar(x + 0.18, v4_vals, 0.36, color=C_V4P, label='V4_pro')
ax5.set_xticks(x); ax5.set_xticklabels(bar_keys, rotation=12)
ax5.set_title("(e) Сводное сравнение метрик")
ax5.grid(alpha=0.3, axis='y'); ax5.legend()
for i, (a, b) in enumerate(zip(v2_vals, v4_vals)):
    ax5.text(i - 0.18, a, f"{a:.3g}", ha='center', va='bottom', fontsize=8)
    ax5.text(i + 0.18, b, f"{b:.3g}", ha='center', va='bottom', fontsize=8)

fig.suptitle("V2 (LK + Kalman) vs V4_pro (Robust LK + Kalman)\n"
             "Идентичный smoother — изолируем эффект feature-блока", fontsize=13, y=0.995)
fig.savefig(f"{PLOT_DIR}/05_thesis_figure.png", dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
from google.colab import files
for f in sorted(os.listdir(PLOT_DIR)):
    if f.endswith('.png'):
        files.download(os.path.join(PLOT_DIR, f))


нейронка

In [ ]:
import torch
import torchvision
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"torch={torch.__version__}  torchvision={torchvision.__version__}  device={DEVICE}")

# Pretrained RAFT-Small (Things + Sintel)
weights    = Raft_Small_Weights.DEFAULT
raft_model = raft_small(weights=weights, progress=False).to(DEVICE).eval()
raft_xform = weights.transforms()

print("RAFT-Small loaded.")
print(f"Params: {sum(p.numel() for p in raft_model.parameters())/1e6:.2f}M")


In [ ]:
import cv2, numpy as np

@torch.no_grad()
def compute_flow(frame1_bgr, frame2_bgr, target_long_side=512):
    """Dense optical flow с RAFT-Small."""
    h0, w0 = frame1_bgr.shape[:2]

    # размер для инференса — кратный 8, длинная сторона = target_long_side
    if w0 >= h0:
        W = target_long_side; H = int(round(h0 / w0 * W))
    else:
        H = target_long_side; W = int(round(w0 / h0 * H))
    H = (H // 8) * 8; W = (W // 8) * 8

    rgb1 = cv2.cvtColor(frame1_bgr, cv2.COLOR_BGR2RGB)
    rgb2 = cv2.cvtColor(frame2_bgr, cv2.COLOR_BGR2RGB)
    img1 = cv2.resize(rgb1, (W, H))
    img2 = cv2.resize(rgb2, (W, H))

    t1 = torch.from_numpy(img1).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t2 = torch.from_numpy(img2).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t1, t2 = raft_xform(t1, t2)
    t1, t2 = t1.to(DEVICE), t2.to(DEVICE)

    flow_pyramid = raft_model(t1, t2)
    flow_low = flow_pyramid[-1][0].cpu().numpy().transpose(1, 2, 0)  # (H, W, 2)

    # масштаб обратно к исходному разрешению
    flow_full = cv2.resize(flow_low, (w0, h0), interpolation=cv2.INTER_LINEAR)
    flow_full[..., 0] *= (w0 / W)
    flow_full[..., 1] *= (h0 / H)
    return flow_full

def affine_from_flow(flow, grid_step=20, mag_threshold=80.0):
    """Из dense flow семплируем регулярную сетку, отбрасываем подозрительно большие."""
    h, w = flow.shape[:2]
    ys = np.arange(0, h, grid_step)
    xs = np.arange(0, w, grid_step)
    Y, X = np.meshgrid(ys, xs, indexing='ij')

    fl = flow[Y, X]                      # (Ny, Nx, 2)
    pts0 = np.stack([X, Y], axis=-1).astype(np.float32).reshape(-1, 2)
    pts1 = (pts0 + fl.reshape(-1, 2)).astype(np.float32)

    mag = np.linalg.norm(fl.reshape(-1, 2), axis=1)
    valid = (mag < mag_threshold) & np.isfinite(mag)
    pts0 = pts0[valid]; pts1 = pts1[valid]
    if len(pts0) < 30:
        return None

    pts0 = pts0.reshape(-1, 1, 2); pts1 = pts1.reshape(-1, 1, 2)
    M, _ = cv2.estimateAffinePartial2D(
        pts0, pts1, method=cv2.RANSAC,
        ransacReprojThreshold=2.0, maxIters=2000,
        confidence=0.999, refineIters=15)
    return M

print("compute_flow + affine_from_flow ready")


In [ ]:
import os, time, cv2, numpy as np

def stabilize_v5(input_path, output_path,
                 side_by_side_path=None,
                 transforms_path=None, raw_transforms_path=None,
                 trajectory_path=None, smoothed_trajectory_path=None,
                 target_long_side=512,
                 grid_step=20,
                 mag_threshold=80.0,
                 smoothing='kalman',
                 smoothing_radius=None,
                 kalman_q=(1e-3, 1e-3, 1e-5),
                 kalman_r=(10.0, 10.0, 1e-2),
                 median_kernel=5, mad_k=4.0,
                 verbose=True):
    """Variant 5: deep optical flow (RAFT-Small) + классическая обвязка."""
    assert os.path.exists(input_path)

    cap = cv2.VideoCapture(input_path)
    w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if smoothing_radius is None:
        smoothing_radius = max(15, int(round(2 * fps)))

    if verbose:
        print(f"[V5] {w}x{h} @ {fps:.2f} fps, {n} frames")
        print(f"[V5] target_long_side={target_long_side}  grid_step={grid_step}  "
              f"smoothing={smoothing}  device={DEVICE}")

    ok, prev = cap.read(); assert ok
    transforms, last_valid = [], np.zeros(3)
    cnt = dict(flow_fail=0, affine_fail=0, ok=0)
    t0 = time.time()

    for i in range(n - 1):
        ok, curr = cap.read()
        if not ok: break

        try:
            flow = compute_flow(prev, curr, target_long_side=target_long_side)
        except Exception as e:
            cnt['flow_fail'] += 1
            transforms.append(last_valid.copy()); prev = curr; continue

        M = affine_from_flow(flow, grid_step=grid_step,
                              mag_threshold=mag_threshold)
        if M is None:
            cnt['affine_fail'] += 1
            transforms.append(last_valid.copy()); prev = curr; continue

        dx, dy, da = decompose_transform(M)
        last_valid = np.array([dx, dy, da])
        transforms.append(last_valid.copy())
        cnt['ok'] += 1
        prev = curr

        if verbose and (i + 1) % 50 == 0:
            elapsed = time.time() - t0
            print(f"  [V5 pass 1] {i+1}/{n-1}  "
                  f"avg={1000*elapsed/(i+1):.1f} ms/frame")

    transforms_raw = np.asarray(transforms, dtype=np.float64)
    if verbose: print(f"[V5 pass 1] done: {cnt}")

    transforms = clean_transforms(transforms_raw, median_kernel, mad_k, verbose)
    trajectory = np.cumsum(transforms, axis=0)

    if smoothing == 'kalman':
        smoothed = kalman_smooth_trajectory(trajectory,
                                            q=kalman_q, r=kalman_r, mode='rts')
    else:
        smoothed = smooth_trajectory(trajectory, radius=smoothing_radius)

    transforms_smooth = transforms + (smoothed - trajectory)

    cap.release(); cap = cv2.VideoCapture(input_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    sbs = None
    if side_by_side_path:
        os.makedirs(os.path.dirname(os.path.abspath(side_by_side_path)) or ".", exist_ok=True)
        sbs = cv2.VideoWriter(side_by_side_path, fourcc, fps, (2 * w, h))

    ok, frame = cap.read(); writer.write(frame)
    if sbs is not None: sbs.write(np.hstack([frame, frame]))

    for i in range(n - 1):
        ok, frame = cap.read()
        if not ok or i >= len(transforms_smooth): break
        dx, dy, da = transforms_smooth[i]
        M = build_transform(dx, dy, da)
        stab = cv2.warpAffine(frame, M, (w, h),
                              flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_REPLICATE)
        writer.write(stab)
        if sbs is not None: sbs.write(np.hstack([frame, stab]))

    cap.release(); writer.release()
    if sbs is not None: sbs.release()

    if transforms_path:         np.save(transforms_path, transforms)
    if raw_transforms_path:     np.save(raw_transforms_path, transforms_raw)
    if trajectory_path:         np.save(trajectory_path, trajectory)
    if smoothed_trajectory_path: np.save(smoothed_trajectory_path, smoothed)

    if verbose: print(f"[V5 done] -> {output_path}")

    return dict(transforms=transforms, transforms_raw=transforms_raw,
                trajectory=trajectory, smoothed_trajectory=smoothed,
                counters=cnt,
                meta=dict(width=w, height=h, fps=fps, n_frames=n,
                          smoothing_radius=smoothing_radius, smoothing=smoothing,
                          target_long_side=target_long_side, grid_step=grid_step,
                          device=DEVICE))

print("stabilize_v5 ready")


In [ ]:
import os

OUTPUT_DIR_V5 = "/content/results_v5"
os.makedirs(OUTPUT_DIR_V5, exist_ok=True)

results_v5 = stabilize_v5(
    input_path              = INPUT_VIDEO,
    output_path             = f"{OUTPUT_DIR_V5}/v5_stabilized.mp4",
    side_by_side_path       = f"{OUTPUT_DIR_V5}/v5_side_by_side.mp4",
    transforms_path         = f"{OUTPUT_DIR_V5}/v5_transforms_clean.npy",
    raw_transforms_path     = f"{OUTPUT_DIR_V5}/v5_transforms_raw.npy",
    trajectory_path         = f"{OUTPUT_DIR_V5}/v5_trajectory.npy",
    smoothed_trajectory_path= f"{OUTPUT_DIR_V5}/v5_smoothed.npy",
    target_long_side=512,            # понизь до 384 если GPU слабый
    grid_step=20,
    mag_threshold=80.0,
    smoothing='kalman',
    kalman_q=(1e-3, 1e-3, 1e-5),
    kalman_r=(10.0, 10.0, 1e-2),
    median_kernel=5, mad_k=4.0,
    verbose=True,
)
print("counters:", results_v5["counters"])

import numpy as np
m_v5 = compute_metrics(
    name="V5 (RAFT-Small + Kalman)",
    original_video=INPUT_VIDEO,
    stabilized_video=f"{OUTPUT_DIR_V5}/v5_stabilized.mp4",
    raw_transforms=np.load(f"{OUTPUT_DIR_V5}/v5_transforms_clean.npy"),
    smoothed_trajectory=np.load(f"{OUTPUT_DIR_V5}/v5_smoothed.npy"),
)

print_comparison([m_v1, m_v2, m_v3, m_v4, m_v4p, m_v5])


In [ ]:
import time
def speed_v5():
    out = "/content/_speed_v5.mp4"
    t0 = time.time()
    res = stabilize_v5(input_path=INPUT_VIDEO, output_path=out, verbose=False)
    dt = time.time() - t0; n = res['meta']['n_frames']
    if os.path.exists(out): os.remove(out)
    return dict(label=f"V5 ({DEVICE})", total_s=round(dt,2),
                ms_per_frame=round(1000*dt/n,2),
                effective_fps=round(n/dt,1))
print(speed_v5())


Algorithms of detection and tracking



In [ ]:
from google.colab import files
uploaded = files.upload()   # откроется диалог выбора файла на твоём компе


In [ ]:
import shutil, os
src = next(iter(uploaded))            # имя загруженного файла
shutil.move(src, "/content/video5.mp4")
print(os.path.getsize("/content/video5.mp4") / 1e6, "MB")


In [ ]:
import os, cv2
path = "/content/video5.mp4"  # или путь из Drive
print("exists:", os.path.exists(path))
print("size MB:", round(os.path.getsize(path) / 1e6, 1))
cap = cv2.VideoCapture(path)
print(f"  {int(cap.get(3))}x{int(cap.get(4))} @ {cap.get(5):.1f} fps, "
      f"{int(cap.get(7))} frames")
cap.release()


начинаем модели

In [ ]:
!pip install ultralytics boxmot -q
print("ok")


In [ ]:
import os, cv2, numpy as np
from scipy.signal import medfilt

FEATURE_PARAMS = dict(maxCorners=500, qualityLevel=0.005,
                      minDistance=12, blockSize=3)
LK_PARAMS = dict(winSize=(21, 21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
FB_THR = 1.0  # forward-backward LK error threshold

def _track_fb(pg_, cg_, pts_):
    if pts_ is None or len(pts_) == 0: return None, None
    cur, sf, _ = cv2.calcOpticalFlowPyrLK(pg_, cg_, pts_, None, **LK_PARAMS)
    if cur is None: return None, None
    back, sb, _ = cv2.calcOpticalFlowPyrLK(cg_, pg_, cur, None, **LK_PARAMS)
    if back is None: return None, None
    good = (sf.flatten() == 1) & (sb.flatten() == 1)
    err = np.linalg.norm(back.reshape(-1, 2) - pts_.reshape(-1, 2), axis=1)
    good &= (err < FB_THR)
    if not np.any(good): return None, None
    return pts_[good], cur[good]

def _decompose(M): return (0., 0., 0.) if M is None else (
    float(M[0, 2]), float(M[1, 2]), float(np.arctan2(M[1, 0], M[0, 0])))

def _build(dx, dy, da):
    return np.array([[np.cos(da), -np.sin(da), dx],
                     [np.sin(da),  np.cos(da), dy]], dtype=np.float64)

def _robust_std(x): return np.median(np.abs(x - np.median(x))) / 0.6745

def _clean(T, kernel=5, k=4.0, verbose=True):
    T = T.copy()
    for i in range(3): T[:, i] = medfilt(T[:, i], kernel_size=kernel)
    for i, name in enumerate(['dx', 'dy', 'da']):
        thr = max(k * _robust_std(T[:, i]), 1e-6)
        n_clip = int(np.sum(np.abs(T[:, i]) > thr))
        T[:, i] = np.clip(T[:, i], -thr, thr)
        if verbose: print(f"  [clean] {name}: thr={thr:.3f}  clipped={n_clip}")
    return T

def _kalman_axis(z, q, r):
    n = len(z); F = np.array([[1., 1.], [0., 1.]])
    Q = q * np.array([[0.25, 0.5], [0.5, 1.0]])
    xp = np.zeros((n, 2)); Pp = np.zeros((n, 2, 2))
    xf = np.zeros((n, 2)); Pf = np.zeros((n, 2, 2))
    xf[0] = [z[0], 0.0]; Pf[0] = np.eye(2) * 10.0
    for k in range(1, n):
        xp[k] = F @ xf[k-1]; Pp[k] = F @ Pf[k-1] @ F.T + Q
        S = Pp[k][0, 0] + r; K = Pp[k][:, 0] / S
        xf[k] = xp[k] + K * (z[k] - xp[k][0])
        Pf[k] = Pp[k] - np.outer(K, Pp[k][0, :])
    xs = np.zeros((n, 2)); xs[-1] = xf[-1]
    for k in range(n - 2, -1, -1):
        try: C = Pf[k] @ F.T @ np.linalg.inv(Pp[k+1])
        except: C = np.zeros((2, 2))
        xs[k] = xf[k] + C @ (xs[k+1] - xp[k+1])
    return xs[:, 0]

def _kalman_traj(traj, q=(1e-3, 1e-3, 1e-5), r=(10., 10., 1e-2)):
    out = np.zeros_like(traj, dtype=np.float64)
    for i in range(3): out[:, i] = _kalman_axis(traj[:, i], q[i], r[i])
    return out

def stabilize_v2(input_path, output_path, min_points=20, verbose=True):
    cap = cv2.VideoCapture(input_path); assert cap.isOpened()
    w = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0; n = int(cap.get(7))
    if verbose: print(f"[V2] {w}x{h} @ {fps:.1f}fps, {n} frames")

    ok, prev = cap.read(); pg = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
    transforms, last = [], np.zeros(3)
    for i in range(n - 1):
        pts = cv2.goodFeaturesToTrack(pg, mask=None, **FEATURE_PARAMS)
        ok, cur = cap.read()
        if not ok: break
        cg = cv2.cvtColor(cur, cv2.COLOR_BGR2GRAY)
        if pts is None or len(pts) < min_points:
            transforms.append(last.copy()); pg = cg; continue
        pgood, cgood = _track_fb(pg, cg, pts)
        if pgood is None or len(pgood) < min_points:
            transforms.append(last.copy()); pg = cg; continue
        M, _ = cv2.estimateAffinePartial2D(pgood, cgood, method=cv2.RANSAC,
            ransacReprojThreshold=2.0, maxIters=3000, confidence=0.999, refineIters=15)
        if M is None:
            transforms.append(last.copy()); pg = cg; continue
        last = np.array(_decompose(M)); transforms.append(last.copy()); pg = cg
        if verbose and (i + 1) % 200 == 0: print(f"  [V2 pass1] {i+1}/{n-1}")

    transforms = np.asarray(transforms, dtype=np.float64)
    transforms = _clean(transforms, verbose=verbose)
    traj = np.cumsum(transforms, axis=0)
    smoothed = _kalman_traj(traj)
    Tsm = transforms + (smoothed - traj)

    cap.release(); cap = cv2.VideoCapture(input_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    ok, frame = cap.read(); writer.write(frame)
    for i in range(n - 1):
        ok, frame = cap.read()
        if not ok or i >= len(Tsm): break
        dx, dy, da = Tsm[i]
        stab = cv2.warpAffine(frame, _build(dx, dy, da), (w, h),
                              flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
        writer.write(stab)
    cap.release(); writer.release()
    if verbose: print(f"[V2 done] -> {output_path}")

print("stabilize_v2 ready")


In [ ]:
import os, json, time, numpy as np, cv2
from ultralytics import YOLO
# Совместимость с разными версиями boxmot:
try:
    from boxmot import OcSort as OCSORT          # v10+
except ImportError:
    try:
        from boxmot.trackers.ocsort.ocsort import OcSort as OCSORT
    except ImportError:
        from boxmot import OCSORT                  # legacy < v10

PERSON_CLASS = 0

def detect_and_track(input_video, output_video,
                     yolo_weights="yolo11n.pt",
                     conf=0.25, iou=0.45, imgsz=640, verbose=True):
    detector = YOLO(yolo_weights)
    tracker = OCSORT(det_thresh=conf, max_age=30, min_hits=3,
                     asso_threshold=0.3, delta_t=3, inertia=0.2, use_byte=False)

    cap = cv2.VideoCapture(input_video); assert cap.isOpened()
    w = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0; n = int(cap.get(7))
    if verbose: print(f"[pipeline] {os.path.basename(input_video)}: "
                      f"{w}x{h} @ {fps:.1f}fps, {n} frames  imgsz={imgsz} conf={conf}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_video, fourcc, fps, (w, h))
    tracks_log, frame_times = [], []
    t_total = time.time(); frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret: break
        t0 = time.time()
        r = detector(frame, classes=[PERSON_CLASS], conf=conf, iou=iou,
                     imgsz=imgsz, verbose=False)[0]
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy()
            cf   = r.boxes.conf.cpu().numpy()
            cl   = r.boxes.cls.cpu().numpy()
            dets = np.column_stack([xyxy, cf, cl]).astype(np.float32)
        else:
            dets = np.empty((0, 6), dtype=np.float32)
        tracks = tracker.update(dets, frame)
        frame_times.append(time.time() - t0)

        annotated = frame.copy(); ft = []
        for t in tracks:
            x1, y1, x2, y2 = map(int, t[:4]); tid = int(t[4])
            tconf = float(t[5]) if len(t) > 5 else 1.0
            color = ((tid * 41) % 255, (tid * 73) % 255, (tid * 113) % 255)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            cv2.putText(annotated, f"ID {tid} {tconf:.2f}", (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
            ft.append({'id': tid, 'bbox': [x1, y1, x2, y2], 'conf': tconf})
        cv2.putText(annotated, f"frame {frame_idx} | persons: {len(tracks)}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        writer.write(annotated)
        tracks_log.append({'frame': frame_idx, 'tracks': ft})
        frame_idx += 1
        if verbose and frame_idx % 100 == 0: print(f"  [pipeline] {frame_idx}/{n}")

    cap.release(); writer.release()

    life = {}
    for e in tracks_log:
        for tr in e['tracks']: life[tr['id']] = life.get(tr['id'], 0) + 1
    jit = []
    for i in range(1, len(tracks_log)):
        prev = {tr['id']: tr['bbox'] for tr in tracks_log[i-1]['tracks']}
        curr = {tr['id']: tr['bbox'] for tr in tracks_log[i]['tracks']}
        for tid, bb in curr.items():
            if tid in prev:
                cx, cy = (bb[0]+bb[2])/2, (bb[1]+bb[3])/2
                px, py = (prev[tid][0]+prev[tid][2])/2, (prev[tid][1]+prev[tid][3])/2
                jit.append(float(np.hypot(cx-px, cy-py)))
    confs_all = [tr['conf'] for e in tracks_log for tr in e['tracks']]
    npf = [len(e['tracks']) for e in tracks_log]
    summary = {
        'video': os.path.basename(input_video),
        'total_frames': frame_idx,
        'unique_track_ids': len(life),
        'mean_track_lifespan_frames': round(float(np.mean(list(life.values()))) if life else 0, 2),
        'max_track_lifespan_frames':  int(max(life.values())) if life else 0,
        'frames_with_tracks':         int(sum(1 for c in npf if c > 0)),
        'avg_tracks_per_frame':       round(float(np.mean(npf)), 3),
        'mean_confidence':            round(float(np.mean(confs_all)) if confs_all else 0.0, 3),
        'mean_bbox_jitter_px':        round(float(np.mean(jit)) if jit else 0.0, 3),
        'p95_bbox_jitter_px':         round(float(np.percentile(jit, 95)) if jit else 0.0, 3),
        'pipeline_ms_per_frame':      round(float(np.mean(frame_times) * 1000), 2),
        'total_runtime_s':            round(time.time() - t_total, 2),
    }
    with open(output_video.replace('.mp4', '_tracks.json'), 'w') as f:
        json.dump({'video': input_video, 'frames': tracks_log,
                   'meta': {'width': w, 'height': h, 'fps': fps, 'n_frames': frame_idx}}, f, indent=2)
    with open(output_video.replace('.mp4', '_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2)
    if verbose:
        print(f"\n[pipeline] {summary['video']} summary:")
        for k, v in summary.items(): print(f"  {k:32s}: {v}")
    return summary, tracks_log

def compare_summaries(s_raw, s_stab):
    keys = ['unique_track_ids', 'mean_track_lifespan_frames',
            'max_track_lifespan_frames', 'avg_tracks_per_frame',
            'mean_confidence', 'mean_bbox_jitter_px',
            'p95_bbox_jitter_px', 'pipeline_ms_per_frame']
    print(f"{'metric':32s}  {'raw':>10s}  {'stab':>10s}  {'Δ (stab-raw)':>14s}")
    print("-" * 72)
    for k in keys:
        a, b = s_raw[k], s_stab[k]; d = b - a
        print(f"{k:32s}  {a:>10.3f}  {b:>10.3f}  {d:>+14.3f}")

def make_side_by_side(va, vb, out):
    cap_a, cap_b = cv2.VideoCapture(va), cv2.VideoCapture(vb)
    w = int(cap_a.get(3)); h = int(cap_a.get(4))
    fps = cap_a.get(5) or 30.0
    writer = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (2 * w, h))
    while True:
        ra, fa = cap_a.read(); rb, fb = cap_b.read()
        if not ra or not rb: break
        cv2.putText(fa, "RAW",  (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        cv2.putText(fb, "STAB", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        writer.write(np.hstack([fa, fb]))
    cap_a.release(); cap_b.release(); writer.release()

print("detect_and_track + compare_summaries + make_side_by_side ready")


In [ ]:
import os
RAW_VIDEO = "/content/video5.mp4"
OUT_DIR   = "/content/results_perception_baseline"
os.makedirs(OUT_DIR, exist_ok=True)

# 1) raw video pipeline
print(">>> RAW")
sum_raw, _ = detect_and_track(
    input_video=RAW_VIDEO,
    output_video=f"{OUT_DIR}/raw_tracked.mp4",
    imgsz=960, conf=0.20,        # для панорамного 1920x960
)

# 2) V2 стабилизация
print("\n>>> STABILIZATION (V2)")
STAB_VIDEO = f"{OUT_DIR}/video5_stab.mp4"
stabilize_v2(input_path=RAW_VIDEO, output_path=STAB_VIDEO, verbose=True)

# 3) stab video pipeline (те же параметры детектора и трекера)
print("\n>>> STAB")
sum_stab, _ = detect_and_track(
    input_video=STAB_VIDEO,
    output_video=f"{OUT_DIR}/stab_tracked.mp4",
    imgsz=960, conf=0.20,
)

# 4) сравнение
print("\n=== Detection + Tracking: raw vs stabilized ===\n")
compare_summaries(sum_raw, sum_stab)

# 5) side-by-side для глазной проверки
make_side_by_side(f"{OUT_DIR}/raw_tracked.mp4",
                  f"{OUT_DIR}/stab_tracked.mp4",
                  f"{OUT_DIR}/sbs_raw_vs_stab.mp4")
print(f"\nside-by-side -> {OUT_DIR}/sbs_raw_vs_stab.mp4")


following

In [ ]:
import os, json, numpy as np, cv2
from collections import Counter

def render_follower(input_video, tracks_json, output_video,
                    mode='longest', target_id=None, manual_id=None,
                    verbose=True):
    """Robust + instrumented follower renderer."""
    print(f"[debug] enter: input={os.path.basename(input_video)}")

    if not os.path.exists(input_video):
        print(f"[debug] FAIL: input_video missing"); return None
    if not os.path.exists(tracks_json):
        print(f"[debug] FAIL: tracks_json missing"); return None

    with open(tracks_json) as f:
        data = json.load(f)
    tracks_log = data.get('frames', data)
    print(f"[debug] loaded {len(tracks_log)} frame entries")

    if target_id is None:
        if mode == 'manual':
            target_id = int(manual_id) if manual_id is not None else None
        else:  # 'longest'
            cnt = Counter()
            for entry in tracks_log:
                for tr in entry.get('tracks', []):
                    cnt[tr['id']] += 1
            target_id = int(cnt.most_common(1)[0][0]) if cnt else None
    print(f"[debug] target_id = {target_id}")
    if target_id is None:
        print(f"[debug] FAIL: no target candidate"); return None

    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        print(f"[debug] FAIL: cannot open input video"); return None
    w   = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0
    cx_frame, cy_frame = w // 2, h // 2
    print(f"[debug] video {w}x{h} @ {fps:.1f} fps")

    os.makedirs(os.path.dirname(os.path.abspath(output_video)) or ".", exist_ok=True)
    writer = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (w, h))
    if not writer.isOpened():
        print(f"[debug] FAIL: VideoWriter not opened (codec/path issue)")
        cap.release(); return None
    print(f"[debug] writer ok -> {output_video}")

    history, present_log, offsets, bbox_areas = [], [], [], []
    n_iter = 0
    try:
        for i, entry in enumerate(tracks_log):
            ret, frame = cap.read()
            if not ret: break
            n_iter += 1

            annotated = frame.copy()
            target = None
            for tr in entry.get('tracks', []):
                if tr['id'] == target_id:
                    target = tr
                else:
                    x1, y1, x2, y2 = (int(v) for v in tr['bbox'])
                    cv2.rectangle(annotated, (x1, y1), (x2, y2), (110, 110, 110), 1)

            if target is not None:
                x1, y1, x2, y2 = (int(v) for v in target['bbox'])
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                history.append((cx, cy))
                offsets.append((cx - cx_frame, cy - cy_frame))
                bbox_areas.append((x2 - x1) * (y2 - y1))
                present_log.append(True)

                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 3)
                cv2.putText(annotated, f"TARGET ID {target_id}",
                            (x1, max(20, y1 - 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.circle(annotated, (cx, cy), 6, (0, 255, 0), -1)
                cv2.line(annotated, (cx_frame, cy_frame), (cx, cy), (0, 255, 255), 2)
            else:
                present_log.append(False)
                cv2.putText(annotated, "TARGET LOST",
                            (max(0, cx_frame - 130), max(20, cy_frame)),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

            cv2.drawMarker(annotated, (cx_frame, cy_frame), (255, 255, 255),
                           cv2.MARKER_CROSS, 20, 1)
            cv2.putText(annotated,
                        f"frame {i} | target {'OK' if target else 'LOST'}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                        (0, 255, 0) if target else (0, 0, 255), 2)
            writer.write(annotated)

            if (i + 1) % 500 == 0:
                print(f"[debug] processed {i+1} frames")

    except BaseException as e:
        import traceback
        print(f"[debug] CRASH inside loop at iter {n_iter}: {type(e).__name__}: {e}")
        traceback.print_exc()
        cap.release(); writer.release()
        return None

    cap.release(); writer.release()
    print(f"[debug] loop done: {n_iter} iters processed")

    n = len(present_log); n_present = sum(present_log)
    lost_segments = []
    in_lost, start = False, 0
    for i, p in enumerate(present_log):
        if not p and not in_lost: in_lost, start = True, i
        elif p and in_lost: in_lost = False; lost_segments.append((start, i - 1))
    if in_lost: lost_segments.append((start, n - 1))
    seg_lens = [b - a + 1 for a, b in lost_segments]
    off_x = np.array([o[0] for o in offsets]) if offsets else np.array([])
    off_y = np.array([o[1] for o in offsets]) if offsets else np.array([])

    summary = {
        'target_id':                    target_id,
        'total_frames':                 n,
        'frames_with_target':           n_present,
        'lock_rate':                    round(n_present / n, 4) if n else 0.0,
        'n_lost_segments':              len(lost_segments),
        'longest_lost_segment_frames':  int(max(seg_lens)) if seg_lens else 0,
        'mean_lost_segment_frames':     round(float(np.mean(seg_lens)), 2) if seg_lens else 0.0,
        'mean_offset_px':               round(float(np.mean(np.hypot(off_x, off_y))), 2) if len(off_x) else 0.0,
        'mean_target_bbox_area':        round(float(np.mean(bbox_areas)), 2) if bbox_areas else 0.0,
    }
    print(f"[debug] summary computed: lock_rate={summary['lock_rate']} n={n}")

    out_json = output_video.replace('.mp4', '_follower_summary.json')
    json.dump(summary, open(out_json, 'w'), indent=2)
    print(f"[debug] saved -> {out_json}")
    print(f"[debug] returning dict")
    return summary

print("render_follower (debug) ready")


In [ ]:
import traceback

def safe_run(label, **kwargs):
    print(f">>> {label}")
    try:
        out = render_follower(**kwargs)
        print(f"[ok] returned: {type(out).__name__}")
        return out
    except Exception as e:
        traceback.print_exc()
        return None

foll_raw  = safe_run("FOLLOWER on RAW",
                     input_video=RAW_VIDEO,
                     tracks_json=f"{OUT_DIR}/raw_tracked_tracks.json",
                     output_video=f"{OUT_DIR}/raw_follower.mp4",
                     mode='longest')

foll_stab = safe_run("FOLLOWER on STAB",
                     input_video=f"{OUT_DIR}/video5_stab.mp4",
                     tracks_json=f"{OUT_DIR}/stab_tracked_tracks.json",
                     output_video=f"{OUT_DIR}/stab_follower.mp4",
                     mode='longest')

if foll_raw and foll_stab:
    compare_follower(foll_raw, foll_stab)
else:
    print("!!! one of follower runs failed — see traceback above")


In [ ]:
import traceback, os, json

def safe_run(label, **kwargs):
    print(f">>> {label}")
    try:
        out = render_follower(**kwargs)
        print(f"[ok] returned type: {type(out).__name__}")
        if isinstance(out, dict):
            print(f"     lock_rate={out.get('lock_rate')}  "
                  f"frames_with_target={out.get('frames_with_target')}")
        return out
    except BaseException as e:
        print(f"[FAILED] {type(e).__name__}: {e}")
        traceback.print_exc()
        return None

foll_raw  = safe_run("RAW",
    input_video=RAW_VIDEO,
    tracks_json=f"{OUT_DIR}/raw_tracked_tracks.json",
    output_video=f"{OUT_DIR}/raw_follower.mp4",
    mode='longest')

foll_stab = safe_run("STAB",
    input_video=f"{OUT_DIR}/video5_stab.mp4",
    tracks_json=f"{OUT_DIR}/stab_tracked_tracks.json",
    output_video=f"{OUT_DIR}/stab_follower.mp4",
    mode='longest')


In [ ]:
for p in [f"{OUT_DIR}/raw_tracked_tracks.json",
          f"{OUT_DIR}/stab_tracked_tracks.json",
          f"{OUT_DIR}/video5_stab.mp4",
          RAW_VIDEO]:
    print(("OK   " if os.path.exists(p) else "MISS ") + p)


добавляем шейк

In [ ]:
import numpy as np, cv2

def add_synthetic_shake(input_path, output_path,
                        trans_std=15.0, rot_std_rad=0.02, seed=42):
    """
    Добавляет высокочастотный random шейк (translation + rotation).
    trans_std=15 px и rot_std=0.02 rad ≈ 1° соответствует среднему «handheld» уровню.
    """
    np.random.seed(seed)
    cap = cv2.VideoCapture(input_path)
    w = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0; n = int(cap.get(7))
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (w, h))
    print(f"[shake] {n} frames @ trans_std={trans_std}px rot_std={rot_std_rad}rad")
    for i in range(n):
        ok, frame = cap.read()
        if not ok: break
        dx = np.random.normal(0, trans_std)
        dy = np.random.normal(0, trans_std * 0.6)
        da = np.random.normal(0, rot_std_rad)
        M = np.array([[np.cos(da), -np.sin(da), dx],
                      [np.sin(da),  np.cos(da), dy]], dtype=np.float64)
        shaky = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        writer.write(shaky)
        if (i + 1) % 1000 == 0: print(f"  [shake] {i+1}/{n}")
    cap.release(); writer.release()
    print(f"[shake] -> {output_path}")

SHAKY_VIDEO = f"{OUT_DIR}/video5_shaky.mp4"
add_synthetic_shake(RAW_VIDEO, SHAKY_VIDEO, trans_std=15.0, rot_std_rad=0.02)


In [ ]:
import os

print(">>> SHAKY-RAW pipeline")
shaky_sum_raw, _ = detect_and_track(
    input_video=SHAKY_VIDEO,
    output_video=f"{OUT_DIR}/shaky_raw_tracked.mp4",
    imgsz=960, conf=0.20,
)

print("\n>>> V2 stabilization on shaky")
SHAKY_STAB = f"{OUT_DIR}/video5_shaky_stab.mp4"
stabilize_v2(input_path=SHAKY_VIDEO, output_path=SHAKY_STAB, verbose=True)

print("\n>>> SHAKY-STAB pipeline")
shaky_sum_stab, _ = detect_and_track(
    input_video=SHAKY_STAB,
    output_video=f"{OUT_DIR}/shaky_stab_tracked.mp4",
    imgsz=960, conf=0.20,
)

print("\n=== Tracking: SHAKY raw vs stabilized ===\n")
compare_summaries(shaky_sum_raw, shaky_sum_stab)

make_side_by_side(f"{OUT_DIR}/shaky_raw_tracked.mp4",
                  f"{OUT_DIR}/shaky_stab_tracked.mp4",
                  f"{OUT_DIR}/sbs_shaky_raw_vs_stab.mp4")
print(f"\nside-by-side -> {OUT_DIR}/sbs_shaky_raw_vs_stab.mp4")


In [ ]:
print(">>> SHAKY FOLLOWER on raw")
shaky_foll_raw = render_follower(
    input_video=SHAKY_VIDEO,
    tracks_json=f"{OUT_DIR}/shaky_raw_tracked_tracks.json",
    output_video=f"{OUT_DIR}/shaky_raw_follower.mp4",
    mode='longest',
)

print("\n>>> SHAKY FOLLOWER on stab")
shaky_foll_stab = render_follower(
    input_video=SHAKY_STAB,
    tracks_json=f"{OUT_DIR}/shaky_stab_tracked_tracks.json",
    output_video=f"{OUT_DIR}/shaky_stab_follower.mp4",
    mode='longest',
)

if shaky_foll_raw and shaky_foll_stab:
    compare_follower(shaky_foll_raw, shaky_foll_stab)
    make_side_by_side(f"{OUT_DIR}/shaky_raw_follower.mp4",
                      f"{OUT_DIR}/shaky_stab_follower.mp4",
                      f"{OUT_DIR}/sbs_shaky_follower.mp4")
    print(f"\nside-by-side -> {OUT_DIR}/sbs_shaky_follower.mp4")
else:
    print("!!! one of follower runs failed — see prints above")


In [ ]:
print("\n" + "=" * 130)
print("FINAL 2x2 SUMMARY: effect of V2 stabilization on perception")
print("                     (calm vs shaky × raw vs stab)")
print("=" * 130)

print(f"\n{'metric':32s}{'CALM video':^32s}    {'SHAKY video':^32s}")
print(f"{'':32s}{'raw':>10s}{'stab':>10s}{'Δ':>10s}    {'raw':>10s}{'stab':>10s}{'Δ':>10s}")
print("-" * 130)

print("\n-- Detection + tracking --")
for k, label in [
    ('mean_bbox_jitter_px',         'BBox jitter (px)'),
    ('p95_bbox_jitter_px',          'BBox jitter p95 (px)'),
    ('unique_track_ids',            '# unique track IDs'),
    ('mean_track_lifespan_frames',  'Mean track lifespan (fr)'),
    ('avg_tracks_per_frame',        'Avg tracks/frame'),
    ('mean_confidence',             'Mean confidence'),
]:
    cr, cs = sum_raw[k], sum_stab[k]
    sr, ss = shaky_sum_raw[k], shaky_sum_stab[k]
    print(f"{label:32s}{cr:>10.3f}{cs:>10.3f}{cs-cr:>+10.3f}    "
          f"{sr:>10.3f}{ss:>10.3f}{ss-sr:>+10.3f}")

print("\n-- Person-following --")
for k, label in [
    ('lock_rate',                   'Lock rate'),
    ('frames_with_target',          'Frames with target'),
    ('n_lost_segments',             '# lost segments'),
    ('longest_lost_segment_frames', 'Longest lost segment (fr)'),
    ('mean_offset_px',              'Mean target offset (px)'),
]:
    cr, cs = foll_raw[k], foll_stab[k]
    sr, ss = shaky_foll_raw[k], shaky_foll_stab[k]
    print(f"{label:32s}{cr:>10.3f}{cs:>10.3f}{cs-cr:>+10.3f}    "
          f"{sr:>10.3f}{ss:>10.3f}{ss-sr:>+10.3f}")

print("\n" + "=" * 130)


реалистичный шейк

In [ ]:
def add_realistic_shake(input_path, output_path,
                        sin_amp_trans=10.0,        # амплитуда низкочастотных колебаний (px)
                        sin_amp_rot=0.015,         # амплитуда колебаний угла (rad ≈ 0.86°)
                        freqs_hz=(2.5, 4.0, 7.0),  # Hz - типичная робот-вибрация
                        white_std=1.5,             # маленький white noise сверху (px)
                        burst_prob=0.015, burst_mag=12.0,  # редкие резкие bump'ы
                        seed=42):
    """Шейк, имитирующий реальную робот-вибрацию:."""
    rng = np.random.default_rng(seed)
    cap = cv2.VideoCapture(input_path)
    w = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0; n = int(cap.get(7))
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (w, h))
    print(f"[realistic-shake] {n} frames, freqs={freqs_hz} Hz")

    phases_x = rng.uniform(0, 2*np.pi, len(freqs_hz))
    phases_y = rng.uniform(0, 2*np.pi, len(freqs_hz))
    phases_a = rng.uniform(0, 2*np.pi, len(freqs_hz))

    for i in range(n):
        ok, frame = cap.read()
        if not ok: break
        t = i / fps
        dx = sum(sin_amp_trans * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, phases_x))
        dy = sum(0.6 * sin_amp_trans * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, phases_y))
        da = sum(sin_amp_rot * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, phases_a))
        dx += rng.normal(0, white_std); dy += rng.normal(0, white_std)
        if rng.random() < burst_prob:
            dx += rng.normal(0, burst_mag)
            dy += rng.normal(0, burst_mag * 0.6)
        M = np.array([[np.cos(da), -np.sin(da), dx],
                      [np.sin(da),  np.cos(da), dy]], dtype=np.float64)
        shaky = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        writer.write(shaky)
        if (i + 1) % 1000 == 0: print(f"  [realistic-shake] {i+1}/{n}")
    cap.release(); writer.release()
    print(f"[realistic-shake] -> {output_path}")

REALSHAKY_VIDEO = f"{OUT_DIR}/video5_realshaky.mp4"
add_realistic_shake(RAW_VIDEO, REALSHAKY_VIDEO)


In [ ]:
print(">>> REALSHAKY-RAW pipeline")
real_sum_raw, _ = detect_and_track(
    input_video=REALSHAKY_VIDEO,
    output_video=f"{OUT_DIR}/realshaky_raw_tracked.mp4",
    imgsz=960, conf=0.20,
)

print("\n>>> V2 on realshaky")
REALSHAKY_STAB = f"{OUT_DIR}/video5_realshaky_stab.mp4"
stabilize_v2(input_path=REALSHAKY_VIDEO, output_path=REALSHAKY_STAB, verbose=True)

print("\n>>> REALSHAKY-STAB pipeline")
real_sum_stab, _ = detect_and_track(
    input_video=REALSHAKY_STAB,
    output_video=f"{OUT_DIR}/realshaky_stab_tracked.mp4",
    imgsz=960, conf=0.20,
)

print("\n=== Tracking: REALSHAKY raw vs stab ===\n")
compare_summaries(real_sum_raw, real_sum_stab)

print("\n>>> REALSHAKY FOLLOWER")
real_foll_raw = render_follower(
    input_video=REALSHAKY_VIDEO,
    tracks_json=f"{OUT_DIR}/realshaky_raw_tracked_tracks.json",
    output_video=f"{OUT_DIR}/realshaky_raw_follower.mp4",
    mode='longest')
real_foll_stab = render_follower(
    input_video=REALSHAKY_STAB,
    tracks_json=f"{OUT_DIR}/realshaky_stab_tracked_tracks.json",
    output_video=f"{OUT_DIR}/realshaky_stab_follower.mp4",
    mode='longest')
if real_foll_raw and real_foll_stab:
    compare_follower(real_foll_raw, real_foll_stab)


In [ ]:
print("\n" + "=" * 160)
print("FINAL SUMMARY: V2 stabilization across three input regimes")
print("=" * 160)
print(f"\n{'metric':30s}"
      f"{'CALM':^36s}    {'WHITE-NOISE SHAKY':^36s}    {'REALISTIC ROBOT SHAKY':^36s}")
print(f"{'':30s}" + (f"{'raw':>10s}{'stab':>10s}{'Δ':>10s}    " * 3))
print("-" * 160)

print("\n-- Detection + tracking --")
for k, label in [
    ('mean_bbox_jitter_px',         'BBox jitter (px)'),
    ('unique_track_ids',            '# unique track IDs'),
    ('mean_track_lifespan_frames',  'Mean track lifespan (fr)'),
]:
    a = sum_raw[k]; b = sum_stab[k]
    c = shaky_sum_raw[k]; d = shaky_sum_stab[k]
    e = real_sum_raw[k]; f = real_sum_stab[k]
    print(f"{label:30s}"
          f"{a:>10.3f}{b:>10.3f}{b-a:>+10.3f}    "
          f"{c:>10.3f}{d:>10.3f}{d-c:>+10.3f}    "
          f"{e:>10.3f}{f:>10.3f}{f-e:>+10.3f}")

print("\n-- Person-following --")
for k, label in [
    ('lock_rate',                   'Lock rate'),
    ('n_lost_segments',             '# lost segments'),
    ('longest_lost_segment_frames', 'Longest lost (fr)'),
]:
    a = foll_raw[k]; b = foll_stab[k]
    c = shaky_foll_raw[k]; d = shaky_foll_stab[k]
    e = real_foll_raw[k]; f = real_foll_stab[k]
    print(f"{label:30s}"
          f"{a:>10.3f}{b:>10.3f}{b-a:>+10.3f}    "
          f"{c:>10.3f}{d:>10.3f}{d-c:>+10.3f}    "
          f"{e:>10.3f}{f:>10.3f}{f-e:>+10.3f}")
print("=" * 160)


Чиним метрику — сделаем консистентную идентификацию того же физического человека на всех 6 вариантах. Идея проста: берём bbox-траекторию reference-target из calm_raw как «истину», и на каждом видео ищем тот трек, чей bbox лучше всего матчится с reference (через IoU). Это снимает зависимость от auto-selection.

In [ ]:
import os, json, numpy as np, cv2
from collections import Counter

def _iou(a, b):
    if a is None or b is None: return 0.0
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0

def consistent_follower(input_video, tracks_json, output_video,
                        reference_traj, iou_threshold=0.2, verbose=True):
    """Рендерит follower-видео, идентифицируя target по reference-траектории."""
    with open(tracks_json) as f:
        tracks_log = json.load(f)['frames']

    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        print(f"[fail] cannot open {input_video}"); return None
    w = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0
    cx_f, cy_f = w // 2, h // 2

    os.makedirs(os.path.dirname(os.path.abspath(output_video)) or ".", exist_ok=True)
    writer = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (w, h))

    present, matched_ids = [], []

    for i, entry in enumerate(tracks_log):
        ok, frame = cap.read()
        if not ok: break
        annotated = frame.copy()

        ref = reference_traj[i] if i < len(reference_traj) else None

        # ищем track с лучшим IoU к reference
        best_iou, best_bbox, best_id = 0.0, None, None
        for tr in entry.get('tracks', []):
            io = _iou(tr['bbox'], ref)
            if io > best_iou and io >= iou_threshold:
                best_iou, best_bbox, best_id = io, tr['bbox'], tr['id']

        # фон: все остальные люди серым
        for tr in entry.get('tracks', []):
            x1, y1, x2, y2 = (int(v) for v in tr['bbox'])
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (110, 110, 110), 1)

        # reference (где должен быть target по эталонной траектории)
        if ref is not None:
            x1, y1, x2, y2 = (int(v) for v in ref)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 255), 1, cv2.LINE_AA)

        if best_bbox is not None:
            x1, y1, x2, y2 = (int(v) for v in best_bbox)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(annotated, f"TARGET ID {best_id} IoU={best_iou:.2f}",
                        (x1, max(20, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            present.append(True); matched_ids.append(best_id)
        else:
            present.append(False); matched_ids.append(None)
            cv2.putText(annotated, "TARGET LOST",
                        (max(0, cx_f - 130), max(20, cy_f)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

        cv2.putText(annotated, f"frame {i}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        writer.write(annotated)

    cap.release(); writer.release()

    n = len(present); n_pres = sum(present)
    valid_ids = [x for x in matched_ids if x is not None]
    distinct_ids = len(set(valid_ids))

    # самый длинный непрерывный run одного и того же ID, матчащего reference
    longest_run, cur_run, cur_id = 0, 0, None
    for tid in matched_ids:
        if tid is None: cur_run, cur_id = 0, None
        elif tid == cur_id: cur_run += 1
        else: cur_run, cur_id = 1, tid
        longest_run = max(longest_run, cur_run)

    # потери
    lost_seg, in_lost, st = [], False, 0
    for i, p in enumerate(present):
        if not p and not in_lost: in_lost, st = True, i
        elif p and in_lost: in_lost = False; lost_seg.append((st, i - 1))
    if in_lost: lost_seg.append((st, n - 1))
    seg_lens = [b - a + 1 for a, b in lost_seg]

    summary = {
        'total_frames':                n,
        'frames_with_target':          n_pres,
        'lock_rate':                   round(n_pres / n, 4) if n else 0.0,
        'distinct_ids_for_target':     distinct_ids,
        'longest_consistent_run':      longest_run,
        'n_lost_segments':             len(lost_seg),
        'longest_lost_segment_frames': int(max(seg_lens)) if seg_lens else 0,
    }
    if verbose: print(f"  -> {summary}")
    return summary

print("consistent_follower ready")


In [ ]:
# 1) Reference target = самый долгий трек в CALM RAW (это "ground truth person")
data_calm = json.load(open(f"{OUT_DIR}/raw_tracked_tracks.json"))
calm_tracks_log = data_calm['frames']

cnt = Counter()
for e in calm_tracks_log:
    for tr in e.get('tracks', []): cnt[tr['id']] += 1
REF_ID = cnt.most_common(1)[0][0]
print(f"Reference target_id (from calm_raw): {REF_ID}, lifespan = {cnt[REF_ID]} frames")

# 2) Reference bbox-trajectory: bbox целевого track ID по кадрам calm_raw
reference_traj = [None] * len(calm_tracks_log)
for i, e in enumerate(calm_tracks_log):
    for tr in e.get('tracks', []):
        if tr['id'] == REF_ID:
            reference_traj[i] = tr['bbox']; break
n_ref = sum(1 for b in reference_traj if b is not None)
print(f"Reference trajectory present in {n_ref}/{len(reference_traj)} frames")

# 3) Прогон consistent_follower на всех вариантах
variants = [
    ('calm_raw',    RAW_VIDEO,                        f"{OUT_DIR}/raw_tracked_tracks.json"),
    ('calm_stab',   f"{OUT_DIR}/video5_stab.mp4",     f"{OUT_DIR}/stab_tracked_tracks.json"),
    ('white_raw',   SHAKY_VIDEO,                      f"{OUT_DIR}/shaky_raw_tracked_tracks.json"),
    ('white_stab',  SHAKY_STAB,                       f"{OUT_DIR}/shaky_stab_tracked_tracks.json"),
    ('real_raw',    REALSHAKY_VIDEO,                  f"{OUT_DIR}/realshaky_raw_tracked_tracks.json"),
    ('real_stab',   REALSHAKY_STAB,                   f"{OUT_DIR}/realshaky_stab_tracked_tracks.json"),
]

cf = {}
for label, video, jsonp in variants:
    print(f"\n>>> consistent follower: {label}")
    cf[label] = consistent_follower(
        input_video=video, tracks_json=jsonp,
        output_video=f"{OUT_DIR}/cf_{label}.mp4",
        reference_traj=reference_traj,
        iou_threshold=0.2,
        verbose=True,
    )


In [ ]:
print("\n" + "=" * 145)
print("CONSISTENT TARGET FOLLOWER (same physical person across all 6 variants, IoU >= 0.2)")
print("=" * 145)
print(f"\n{'metric':35s}"
      f"{'CALM':^33s}    {'WHITE-NOISE SHAKY':^33s}    {'REALISTIC ROBOT SHAKY':^33s}")
print(f"{'':35s}" + (f"{'raw':>10s}{'stab':>10s}{'Δ':>10s}    " * 3))
print("-" * 145)

rows = [
    ('lock_rate',                   'Lock rate (consistent target)'),
    ('distinct_ids_for_target',     '# distinct IDs needed'),
    ('longest_consistent_run',      'Longest single-ID run (fr)'),
    ('n_lost_segments',             '# lost segments'),
    ('longest_lost_segment_frames', 'Longest lost segment (fr)'),
]
for k, label in rows:
    a = cf['calm_raw'][k];   b = cf['calm_stab'][k]
    c = cf['white_raw'][k];  d = cf['white_stab'][k]
    e = cf['real_raw'][k];   f = cf['real_stab'][k]
    print(f"{label:35s}"
          f"{a:>10.3f}{b:>10.3f}{b-a:>+10.3f}    "
          f"{c:>10.3f}{d:>10.3f}{d-c:>+10.3f}    "
          f"{e:>10.3f}{f:>10.3f}{f-e:>+10.3f}")
print("=" * 145)


еще прогон

In [ ]:
def add_mild_realistic_shake(input_path, output_path,
                             sin_amp_trans=4.0, sin_amp_rot=0.008,
                             freqs_hz=(1.5, 2.5),
                             white_std=0.8, seed=42):
    """
    Мягкая робот-вибрация: «гладкий пол склада, плавная езда, моторы».
    Peak shake ~6-10 px, freqs 1.5-2.5 Hz — точно в зоне V2's Kalman.
    """
    rng = np.random.default_rng(seed)
    cap = cv2.VideoCapture(input_path)
    w = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0; n = int(cap.get(7))
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (w, h))
    print(f"[mild-shake] {n} frames, freqs={freqs_hz} Hz, sin_amp={sin_amp_trans}")
    px = rng.uniform(0, 2*np.pi, len(freqs_hz))
    py = rng.uniform(0, 2*np.pi, len(freqs_hz))
    pa = rng.uniform(0, 2*np.pi, len(freqs_hz))
    for i in range(n):
        ok, frame = cap.read()
        if not ok: break
        t = i / fps
        dx = sum(sin_amp_trans * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, px))
        dy = sum(0.6 * sin_amp_trans * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, py))
        da = sum(sin_amp_rot * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, pa))
        dx += rng.normal(0, white_std); dy += rng.normal(0, white_std)
        M = np.array([[np.cos(da), -np.sin(da), dx],
                      [np.sin(da),  np.cos(da), dy]], dtype=np.float64)
        writer.write(cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE))
        if (i + 1) % 1000 == 0: print(f"  [mild-shake] {i+1}/{n}")
    cap.release(); writer.release()
    print(f"[mild-shake] -> {output_path}")

MILD_VIDEO = f"{OUT_DIR}/video5_mildshaky.mp4"
add_mild_realistic_shake(RAW_VIDEO, MILD_VIDEO)

# полный пайплайн
print("\n>>> MILD-RAW pipeline")
mild_sum_raw, _ = detect_and_track(input_video=MILD_VIDEO,
    output_video=f"{OUT_DIR}/mild_raw_tracked.mp4", imgsz=960, conf=0.20)

print("\n>>> V2 на mild")
MILD_STAB = f"{OUT_DIR}/video5_mildshaky_stab.mp4"
stabilize_v2(input_path=MILD_VIDEO, output_path=MILD_STAB, verbose=True)

print("\n>>> MILD-STAB pipeline")
mild_sum_stab, _ = detect_and_track(input_video=MILD_STAB,
    output_video=f"{OUT_DIR}/mild_stab_tracked.mp4", imgsz=960, conf=0.20)

# consistent follower
print("\n>>> consistent followers on mild")
cf['mild_raw'] = consistent_follower(
    input_video=MILD_VIDEO,
    tracks_json=f"{OUT_DIR}/mild_raw_tracked_tracks.json",
    output_video=f"{OUT_DIR}/cf_mild_raw.mp4",
    reference_traj=reference_traj, iou_threshold=0.2)
cf['mild_stab'] = consistent_follower(
    input_video=MILD_STAB,
    tracks_json=f"{OUT_DIR}/mild_stab_tracked_tracks.json",
    output_video=f"{OUT_DIR}/cf_mild_stab.mp4",
    reference_traj=reference_traj, iou_threshold=0.2)


In [ ]:
#финальная таблица
print("\n" + "=" * 175)
print("FINAL: V2 stabilization across 4 input regimes (consistent-target follower)")
print("=" * 175)
print(f"\n{'metric':32s}"
      f"{'CALM':^28s}    {'MILD ROBOT':^28s}    {'STRONG ROBOT':^28s}    {'WHITE NOISE':^28s}")
print(f"{'':32s}" + (f"{'raw':>9s}{'stab':>9s}{'Δ':>9s}    " * 4))
print("-" * 175)

rows = [
    ('lock_rate',                   'Lock rate'),
    ('distinct_ids_for_target',     '# distinct IDs'),
    ('longest_consistent_run',      'Longest single-ID run'),
    ('n_lost_segments',             '# lost segments'),
]
for k, label in rows:
    line = f"{label:32s}"
    for r1, r2 in [('calm_raw','calm_stab'), ('mild_raw','mild_stab'),
                   ('real_raw','real_stab'), ('white_raw','white_stab')]:
        a = cf[r1][k]; b = cf[r2][k]
        line += f"{a:>9.2f}{b:>9.2f}{b-a:>+9.2f}    "
    print(line)


In [ ]:
def add_medium_shake(input_path, output_path,
                    sin_amp_trans=8.0, sin_amp_rot=0.012,
                    freqs_hz=(1.0, 2.0),
                    white_std=0.5, seed=42):
    """
    Sweet-spot shake: чисто низкочастотная плавная качка.
    Peak ~12-15 px, freqs 1-2 Hz — точно в зоне V2's Kalman.
    """
    rng = np.random.default_rng(seed)
    cap = cv2.VideoCapture(input_path)
    w = int(cap.get(3)); h = int(cap.get(4))
    fps = cap.get(5) or 30.0; n = int(cap.get(7))
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'),
                             fps, (w, h))
    print(f"[medium-shake] {n} frames, freqs={freqs_hz} Hz, sin_amp={sin_amp_trans}")
    px = rng.uniform(0, 2*np.pi, len(freqs_hz))
    py = rng.uniform(0, 2*np.pi, len(freqs_hz))
    pa = rng.uniform(0, 2*np.pi, len(freqs_hz))
    for i in range(n):
        ok, frame = cap.read()
        if not ok: break
        t = i / fps
        dx = sum(sin_amp_trans * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, px))
        dy = sum(0.6*sin_amp_trans * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, py))
        da = sum(sin_amp_rot * np.sin(2*np.pi*f*t + p) for f, p in zip(freqs_hz, pa))
        dx += rng.normal(0, white_std); dy += rng.normal(0, white_std)
        M = np.array([[np.cos(da), -np.sin(da), dx],
                      [np.sin(da),  np.cos(da), dy]], dtype=np.float64)
        writer.write(cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE))
        if (i + 1) % 1000 == 0: print(f"  [medium-shake] {i+1}/{n}")
    cap.release(); writer.release()
    print(f"[medium-shake] -> {output_path}")

MED_VIDEO = f"{OUT_DIR}/video5_medshaky.mp4"
add_medium_shake(RAW_VIDEO, MED_VIDEO)

# полный пайплайн
print("\n>>> MED-RAW pipeline")
med_sum_raw, _ = detect_and_track(input_video=MED_VIDEO,
    output_video=f"{OUT_DIR}/med_raw_tracked.mp4", imgsz=960, conf=0.20)

print("\n>>> V2 на medium")
MED_STAB = f"{OUT_DIR}/video5_medshaky_stab.mp4"
stabilize_v2(input_path=MED_VIDEO, output_path=MED_STAB, verbose=True)

print("\n>>> MED-STAB pipeline")
med_sum_stab, _ = detect_and_track(input_video=MED_STAB,
    output_video=f"{OUT_DIR}/med_stab_tracked.mp4", imgsz=960, conf=0.20)

# consistent follower
cf['med_raw'] = consistent_follower(
    input_video=MED_VIDEO,
    tracks_json=f"{OUT_DIR}/med_raw_tracked_tracks.json",
    output_video=f"{OUT_DIR}/cf_med_raw.mp4",
    reference_traj=reference_traj, iou_threshold=0.2)
cf['med_stab'] = consistent_follower(
    input_video=MED_STAB,
    tracks_json=f"{OUT_DIR}/med_stab_tracked_tracks.json",
    output_video=f"{OUT_DIR}/cf_med_stab.mp4",
    reference_traj=reference_traj, iou_threshold=0.2)

# финальная таблица 5×2
print("\n" + "=" * 200)
print("FINAL: V2 across 5 shake regimes")
print("=" * 200)
print(f"\n{'metric':32s}"
      f"{'CALM':^28s}    {'MILD':^28s}    {'MEDIUM':^28s}    {'STRONG':^28s}    {'WHITE NOISE':^28s}")
print(f"{'':32s}" + (f"{'raw':>9s}{'stab':>9s}{'Δ':>9s}    " * 5))
print("-" * 200)
for k, label in [
    ('lock_rate', 'Lock rate'),
    ('distinct_ids_for_target', '# distinct IDs'),
    ('longest_consistent_run', 'Longest single-ID run'),
    ('n_lost_segments', '# lost segments'),
]:
    line = f"{label:32s}"
    for r1, r2 in [('calm_raw','calm_stab'), ('mild_raw','mild_stab'),
                   ('med_raw','med_stab'), ('real_raw','real_stab'),
                   ('white_raw','white_stab')]:
        a = cf[r1][k]; b = cf[r2][k]
        line += f"{a:>9.2f}{b:>9.2f}{b-a:>+9.2f}    "
    print(line)
